# Dataset Construction and Feature Engineering

This notebook combines the cleaned tennis match data with the collected interview data.

The pipeline constructs one record per player-tournament, maps player and tournament names across the two data sources, attaches pre-tournament interviews, creates modeling features based on current and previous tournament performance, and defines the final prediction target.

In [ ]:
from pathlib import Path
import json
import re
import time
import unicodedata
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
import shutil

# ============================================================
# Project paths
# ============================================================

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "data_code":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
CLEAN_MATCHES_DIR = DATA_DIR / "clean_matches"
INTERVIEW_DATA_DIR = DATA_DIR / "interviews"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Input files
# ============================================================

ATP_CLEAN_MATCHES_FILE = CLEAN_MATCHES_DIR / "atp_matches_all_clean.csv"
WTA_CLEAN_MATCHES_FILE = CLEAN_MATCHES_DIR / "wta_matches_all_clean.csv"
PLAYER_TOURNAMENT_FILE = PROCESSED_DATA_DIR / "player_tournament_data.csv"
INTERVIEWS_FILE = INTERVIEW_DATA_DIR / "pre_match_interviews_2020_2026.csv"
INTERVIEW_STATUS_FILE = INTERVIEW_DATA_DIR / "tournament_interview_status_2020_2026.csv"

# ============================================================
# Intermediate and output files
# ============================================================

MERGED_DATASET_FILE = PROCESSED_DATA_DIR / "player_tournament_interviews_2020_2026.csv"
FEATURES_DATASET_FILE = PROCESSED_DATA_DIR / "player_tournament_interview_features_2020_2026.csv"
PREV3_FULL_DATASET_FILE = PROCESSED_DATA_DIR / "player_tournament_interview_features_with_prev3_history.csv"
INTERVIEWS_ONLY_FILE = PROCESSED_DATA_DIR / "player_tournament_interview_features_with_prev3_history_interviews_only.csv"
INTERVIEWS_FIXED_FILE = PROCESSED_DATA_DIR / "player_tournament_interview_features_with_prev3_history_interviews_only_fix_missing_questions.csv"
FINAL_INTERVIEWS_FILE = PROCESSED_DATA_DIR / "player_tournament_interview_features_with_prev3_history_interviews_only_fix_missing_questions_with_interview_features.csv"

# ============================================================
# final dataset file
# ============================================================
FINAL_DATASET = PROCESSED_DATA_DIR / "player_tournament_interview_dataset.csv"

In [ ]:
atp_matches_df_clean = pd.read_csv(ATP_CLEAN_MATCHES_FILE)
wta_matches_df_clean = pd.read_csv(WTA_CLEAN_MATCHES_FILE)
all_matches_df_clean = pd.concat([atp_matches_df_clean, wta_matches_df_clean], ignore_index=True)

In [ ]:
# ============================================================
# Load data
# ============================================================

player_tournament_df = pd.read_csv(
    PLAYER_TOURNAMENT_FILE,
    parse_dates=["tourney_date"],
    low_memory=False
)

status_df = pd.read_csv(
    INTERVIEW_STATUS_FILE,
    low_memory=False
)

interviews_df = pd.read_csv(
    INTERVIEWS_FILE,
    parse_dates=["interview_date"],
    low_memory=False
)

print("Player-tournament rows:", len(player_tournament_df))
print("Status rows:", len(status_df))
print("Saved pre-tournament rows:", len(interviews_df))

## Player Name Mapping

In [ ]:
# ============================================================
# Helpers
# ============================================================

def simple_name(value):
    if pd.isna(value):
        return None
    return str(value).strip().lower()

def normalize_name(value):
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    # Remove accents
    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )

    # Treat punctuation, hyphens, apostrophes, etc. as spaces
    value = re.sub(r"[^a-z0-9]+", " ", value)

    # Remove extra spaces
    value = re.sub(r"\s+", " ", value).strip()

    return value if value else None

def reverse_two_part_name(value):
    if pd.isna(value):
        return None

    parts = str(value).split()

    if len(parts) != 2:
        return None

    return f"{parts[1]} {parts[0]}"

# ============================================================
# Prepare unique players from tournament data
# ============================================================

tournament_players_df = (
    player_tournament_df[
        ["player_id", "player_name"]
    ]
    .drop_duplicates()
    .copy()
)

tournament_players_df["simple_name"] = (
    tournament_players_df["player_name"]
    .apply(simple_name)
)

tournament_players_df["normalized_name"] = (
    tournament_players_df["player_name"]
    .apply(normalize_name)
)

# ============================================================
# Prepare unique interview player names
# ============================================================

interview_players_df = (
    interviews_df[
        ["player"]
    ]
    .drop_duplicates()
    .copy()
)

interview_players_df["simple_name"] = (
    interview_players_df["player"]
    .apply(simple_name)
)

interview_players_df["normalized_name"] = (
    interview_players_df["player"]
    .apply(normalize_name)
)


interview_players_df["reversed_simple_name"] = (
    interview_players_df["simple_name"]
    .apply(reverse_two_part_name)
)

interview_players_df["reversed_normalized_name"] = (
    interview_players_df["normalized_name"]
    .apply(reverse_two_part_name)
)

# Final result columns
interview_players_df["matched_player_id"] = pd.NA
interview_players_df["matched_player_name"] = pd.NA
interview_players_df["match_method"] = "unmatched"

# ============================================================
# Build lookup tables and detect conflicts
# ============================================================

def build_lookup(players_df, key_column):
    grouped = (
        players_df.groupby(key_column, dropna=False)
        .agg(
            num_players=("player_id", "nunique"),
            player_ids=("player_id", lambda x: sorted(set(x.dropna()))),
            player_names=("player_name", lambda x: sorted(set(x.dropna())))
        )
        .reset_index()
    )

    conflicts = grouped[
        grouped["num_players"] > 1
    ].copy()

    safe = grouped[
        grouped["num_players"] == 1
    ].copy()

    return safe, conflicts

simple_lookup_df, simple_conflicts_df = build_lookup(
    tournament_players_df,
    "simple_name"
)

normalized_lookup_df, normalized_conflicts_df = build_lookup(
    tournament_players_df,
    "normalized_name"
)

print("Direct-name conflicts:", len(simple_conflicts_df))
if not simple_conflicts_df.empty:
    display(simple_conflicts_df)

print("Normalized-name conflicts:", len(normalized_conflicts_df))
if not normalized_conflicts_df.empty:
    display(normalized_conflicts_df)

# ============================================================
# Stage 1 - Direct match
# ============================================================

direct_matches_df = (
    interview_players_df[
        ["player", "simple_name"]
    ]
    .merge(
        simple_lookup_df[
            ["simple_name", "player_ids", "player_names"]
        ],
        on="simple_name",
        how="left"
    )
)

direct_match_mask = (
    direct_matches_df["player_ids"].notna()
)

for idx in direct_matches_df[direct_match_mask].index:
    original_player = direct_matches_df.loc[idx, "player"]
    player_id = direct_matches_df.loc[idx, "player_ids"][0]
    player_name = direct_matches_df.loc[idx, "player_names"][0]
    target_mask = (interview_players_df["player"] == original_player)

    interview_players_df.loc[
        target_mask,
        "matched_player_id"
    ] = player_id

    interview_players_df.loc[
        target_mask,
        "matched_player_name"
    ] = player_name

    interview_players_df.loc[
        target_mask,
        "match_method"
    ] = "direct"

print("\nDirect matches:", (interview_players_df["match_method"] == "direct").sum())

# ============================================================
# Stage 2 - normalized matching for unmatched names
# ============================================================

unmatched_mask = (interview_players_df["match_method"] == "unmatched")

normalized_candidates_df = (
    interview_players_df.loc[
        unmatched_mask,
        ["player", "normalized_name"]
    ]
    .merge(
        normalized_lookup_df[
            ["normalized_name", "player_ids", "player_names"]
        ],
        on="normalized_name",
        how="left"
    )
)

normalized_match_mask = normalized_candidates_df["player_ids"].notna()

for idx in normalized_candidates_df[normalized_match_mask].index:
    original_player = normalized_candidates_df.loc[idx, "player"]
    player_id = normalized_candidates_df.loc[idx, "player_ids"][0]
    player_name = normalized_candidates_df.loc[idx, "player_names"][0]
    target_mask = (interview_players_df["player"] == original_player)
    interview_players_df.loc[target_mask, "matched_player_id"] = player_id
    interview_players_df.loc[target_mask, "matched_player_name"] = player_name
    interview_players_df.loc[target_mask, "match_method"] = "normalized"

print("Normalization matches:", (interview_players_df["match_method"] == "normalized").sum())

# ============================================================
# Stage 3 - reversed-name matching
# ============================================================

unmatched_mask = interview_players_df["match_method"] == "unmatched"

# Check conflicts for reversed simple names
reverse_simple_conflicts_df = (
    interview_players_df.loc[
        unmatched_mask,
        ["player", "reversed_simple_name"]
    ]
    .merge(
        simple_conflicts_df[
            ["simple_name", "num_players", "player_ids", "player_names"]
        ],
        left_on="reversed_simple_name",
        right_on="simple_name",
        how="inner"
    )
)

print("Reversed simple-name conflicts:", len(reverse_simple_conflicts_df))

if not reverse_simple_conflicts_df.empty:
    display(reverse_simple_conflicts_df)
    raise ValueError("Some reversed simple names match multiple tournament players.")

# Try reversed simple name
reverse_simple_candidates_df = (
    interview_players_df.loc[
        unmatched_mask,
        ["player", "reversed_simple_name"]
    ]
    .merge(
        simple_lookup_df[
            ["simple_name", "player_ids", "player_names"]
        ],
        left_on="reversed_simple_name",
        right_on="simple_name",
        how="left"
    )
)

reverse_simple_match_mask = reverse_simple_candidates_df["player_ids"].notna()

for idx in reverse_simple_candidates_df[reverse_simple_match_mask].index:
    original_player = reverse_simple_candidates_df.loc[idx, "player"]
    player_id = reverse_simple_candidates_df.loc[idx, "player_ids"][0]
    player_name = reverse_simple_candidates_df.loc[idx, "player_names"][0]
    target_mask = interview_players_df["player"] == original_player
    interview_players_df.loc[target_mask, "matched_player_id"] = player_id
    interview_players_df.loc[target_mask, "matched_player_name"] = player_name
    interview_players_df.loc[target_mask, "match_method"] = "reversed_simple"

# Only names still unmatched continue to reversed normalized name
unmatched_mask = interview_players_df["match_method"] == "unmatched"

reverse_normalized_conflicts_df = (
    interview_players_df.loc[
        unmatched_mask,
        ["player", "reversed_normalized_name"]
    ]
    .merge(
        normalized_conflicts_df[
            ["normalized_name", "num_players", "player_ids", "player_names"]
        ],
        left_on="reversed_normalized_name",
        right_on="normalized_name",
        how="inner"
    )
)

print("Reversed normalized-name conflicts:", len(reverse_normalized_conflicts_df))

if not reverse_normalized_conflicts_df.empty:
    display(reverse_normalized_conflicts_df)
    raise ValueError("Some reversed normalized names match multiple tournament players.")

reverse_normalized_candidates_df = (
    interview_players_df.loc[
        unmatched_mask,
        ["player", "reversed_normalized_name"]
    ]
    .merge(
        normalized_lookup_df[
            ["normalized_name", "player_ids", "player_names"]
        ],
        left_on="reversed_normalized_name",
        right_on="normalized_name",
        how="left"
    )
)

reverse_normalized_match_mask = reverse_normalized_candidates_df["player_ids"].notna()

for idx in reverse_normalized_candidates_df[reverse_normalized_match_mask].index:
    original_player = reverse_normalized_candidates_df.loc[idx, "player"]
    player_id = reverse_normalized_candidates_df.loc[idx, "player_ids"][0]
    player_name = reverse_normalized_candidates_df.loc[idx, "player_names"][0]
    target_mask = interview_players_df["player"] == original_player
    interview_players_df.loc[target_mask, "matched_player_id"] = player_id
    interview_players_df.loc[target_mask, "matched_player_name"] = player_name
    interview_players_df.loc[target_mask, "match_method"] = "reversed_normalized"

print("Reversed simple-name matches:", (interview_players_df["match_method"] == "reversed_simple").sum())
print("Reversed normalized-name matches:", (interview_players_df["match_method"] == "reversed_normalized").sum())

# ============================================================
# Final summary
# ============================================================

print("\nFinal player-name matching summary:")
print(interview_players_df["match_method"].value_counts())

# ============================================================
# Show matches created by normalization
# ============================================================

print("\nMatches created by normalization:")

display(
    interview_players_df[
        interview_players_df["match_method"]
        == "normalized"
    ][
        [
            "player",
            "simple_name",
            "normalized_name",
            "matched_player_name",
            "matched_player_id"
        ]
    ]
)

# ============================================================
# Show matches created by reversing
# ============================================================

print("\nMatches created by reversing:")

display(
    interview_players_df[
        interview_players_df["match_method"].isin(
            ["reversed_simple", "reversed_normalized"]
        )
    ][
        [
            "player",
            "simple_name",
            "normalized_name",
            "reversed_simple_name",
            "reversed_normalized_name",
            "match_method",
            "matched_player_name",
            "matched_player_id"
        ]
    ]
)

# ============================================================
# Still unmatched
# ============================================================

unmatched_player_names_df = (
    interview_players_df[
        interview_players_df["match_method"]
        == "unmatched"
    ]
    .copy()
)

print(
    "\nStill unmatched player names:",
    len(unmatched_player_names_df)
)

display(
    unmatched_player_names_df[
        [
            "player",
            "simple_name",
            "normalized_name",
            "reversed_normalized_name"
        ]
    ]
)

# ============================================================
# Check for ambiguous player mappings
# ============================================================

matched_conflicts_df = (
    interview_players_df[
        interview_players_df["match_method"]
        != "unmatched"
    ]
    .groupby("player")
    .agg(
        num_matched_ids=(
            "matched_player_id",
            "nunique"
        ),
        matched_ids=(
            "matched_player_id",
            lambda x: sorted(set(x.dropna()))
        ),
        matched_names=(
            "matched_player_name",
            lambda x: sorted(set(x.dropna()))
        )
    )
    .reset_index()
)

matched_conflicts_df = matched_conflicts_df[matched_conflicts_df["num_matched_ids"] > 1]

print("\nFinal matching conflicts:", len(matched_conflicts_df))

if not matched_conflicts_df.empty:
    display(matched_conflicts_df)
    raise ValueError("Some interview names were matched to multiple players.")

In [ ]:
# ============================================================
# Validate final player mapping
# ============================================================

resolved_player_mapping_df = (
    interview_players_df[
        interview_players_df["match_method"] != "unmatched"
    ][
        [
            "player",
            "matched_player_id",
            "matched_player_name",
            "match_method"
        ]
    ]
    .drop_duplicates()
    .copy()
)

# 1. One interview name must not map to multiple player IDs
source_name_conflicts_df = (
    resolved_player_mapping_df.groupby("player")
    .agg(
        num_player_ids=("matched_player_id", "nunique"),
        player_ids=("matched_player_id", lambda x: sorted(set(x.dropna()))),
        matched_names=("matched_player_name", lambda x: sorted(set(x.dropna()))),
        methods=("match_method", lambda x: sorted(set(x.dropna())))
    )
    .reset_index()
)

source_name_conflicts_df = source_name_conflicts_df[source_name_conflicts_df["num_player_ids"] > 1].copy()

print("Source-name mapping conflicts:", len(source_name_conflicts_df))

if not source_name_conflicts_df.empty:
    display(source_name_conflicts_df)
    raise ValueError("Some interview player names map to multiple player IDs.")

# 2. One matched player name must not belong to multiple player IDs
target_name_conflicts_df = (
    resolved_player_mapping_df.groupby("matched_player_name")
    .agg(
        num_player_ids=("matched_player_id", "nunique"),
        player_ids=("matched_player_id", lambda x: sorted(set(x.dropna()))),
        source_names=("player", lambda x: sorted(set(x.dropna())))
    )
    .reset_index()
)

target_name_conflicts_df = target_name_conflicts_df[target_name_conflicts_df["num_player_ids"] > 1].copy()

print("Target-name conflicts:", len(target_name_conflicts_df))

if not target_name_conflicts_df.empty:
    display(target_name_conflicts_df)
    raise ValueError("Some matched player names belong to multiple player IDs.")

# 3. Build the final mapping only after all checks pass
PLAYER_NAME_MAPPING = dict(
    zip(
        resolved_player_mapping_df["player"],
        resolved_player_mapping_df["matched_player_name"]
    )
)

print("Final player-name mappings:", len(PLAYER_NAME_MAPPING))

# Check whether multiple interview names map to the same player
multiple_interview_names_per_player_df = (
    interview_players_df[
        interview_players_df["match_method"] != "unmatched"
    ]
    .groupby(
        ["matched_player_id", "matched_player_name"],
        dropna=False
    )
    .agg(
        num_interview_names=("player", "nunique"),
        interview_names=("player", lambda x: sorted(set(x.dropna()))),
        match_methods=("match_method", lambda x: sorted(set(x.dropna())))
    )
    .reset_index()
)

multiple_interview_names_per_player_df = (
    multiple_interview_names_per_player_df[
        multiple_interview_names_per_player_df["num_interview_names"] > 1
    ]
)

print("Players with multiple interview-name variants:", len(multiple_interview_names_per_player_df))

if not multiple_interview_names_per_player_df.empty:
    display(multiple_interview_names_per_player_df)

In [ ]:
player_name_mapping_df = (
    interview_players_df[
        interview_players_df["match_method"] != "unmatched"
    ][
        ["player", "matched_player_name", "match_method"]
    ]
    .drop_duplicates()
    .copy()
)

print("Player name mappings:", len(player_name_mapping_df))
display(player_name_mapping_df)

In [ ]:
player_mapping_conflicts_df = (
    player_name_mapping_df.groupby("player")
    .agg(
        num_matches=("matched_player_name", "nunique"),
        matched_names=("matched_player_name", lambda x: sorted(set(x)))
    )
    .reset_index()
)

player_mapping_conflicts_df = player_mapping_conflicts_df[
    player_mapping_conflicts_df["num_matches"] > 1
]

print("Player mapping conflicts:", len(player_mapping_conflicts_df))

if not player_mapping_conflicts_df.empty:
    display(player_mapping_conflicts_df)
    raise ValueError("Some interview player names map to multiple players.")

In [ ]:
PLAYER_NAME_MAPPING = dict(
    zip(
        player_name_mapping_df["player"],
        player_name_mapping_df["matched_player_name"]
    )
)

## Tournament Name Mapping

In [ ]:
# ============================================================
# Normalize tournament names
# ============================================================

def normalize_tournament_name(value):
    if pd.isna(value):
        return None
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value if value else None

# ============================================================
# Known tournament-name mapping
# ============================================================

TOURNAMENT_CANONICAL_MAPPING = {
    "THE CHAMPIONSHIPS": "Wimbledon",
    "WIMBLEDON": "Wimbledon",

    "BNP PARIBAS OPEN": "Indian Wells",
    "INDIAN WELLS": "Indian Wells",
    "INDIAN WELLS MASTERS": "Indian Wells",

    "INTERNAZIONALI BNL D'ITALIA": "Rome",
    "ROME": "Rome",
    "ROME MASTERS": "Rome",

    "MIAMI OPEN PRESENTED BY ITAÚ": "Miami",
    "MIAMI": "Miami",
    "MIAMI MASTERS": "Miami",

    "DUBAI DUTY FREE TENNIS CHAMPIONSHIPS": "Dubai",
    "DUBAI": "Dubai",

    "WESTERN & SOUTHERN OPEN": "Cincinnati",
    "CINCINNATI": "Cincinnati",
    "CINCINNATI MASTERS": "Cincinnati",

    "MUTUA MADRID OPEN": "Madrid",
    "MADRID": "Madrid",
    "MADRID MASTERS": "Madrid",

    "NATIONAL BANK OPEN": "Canada Masters",
    "OMNIUM BANQUE NATIONALE": "Canada Masters",
    "CANADA MASTERS": "Canada Masters",

    "ROLEX MONTE-CARLO MASTERS": "Monte Carlo Masters",
    "MONTE CARLO MASTERS": "Monte Carlo Masters",

    "ROLEX PARIS MASTERS": "Paris Masters",
    "PARIS MASTERS": "Paris Masters",

    "CHINA OPEN": "Beijing",
    "BEIJING": "Beijing",

    "CITI OPEN": "Washington",
    "MUBADALA CITI DC OPEN": "Washington",
    "WASHINGTON": "Washington",

    "ROLEX SHANGHAI MASTERS": "Shanghai Masters",
    "SHANGHAI MASTERS": "Shanghai Masters",

    "QATAR EXXONMOBIL OPEN": "Doha",
    "QATAR TOTAL OPEN": "Doha",
    "QATAR TOTAL ENERGIES OPEN": "Doha",
    "DOHA": "Doha",

    "PORSCHE TENNIS GRAND PRIX": "Stuttgart",
    "STUTTGART": "Stuttgart",

    "CINCH CHAMPIONSHIPS": "Queen's Club",
    "HSBC CHAMPIONSHIPS": "Queen's Club",
    "QUEEN'S CLUB": "Queen's Club",

    "ROTHESAY INTERNATIONAL": "Eastbourne",
    "VIKING INTERNATIONAL": "Eastbourne",
    "EASTBOURNE": "Eastbourne",

    "BRISBANE INTERNATIONAL": "Brisbane",
    "BRISBANE": "Brisbane",

    "SYDNEY TENNIS CLASSIC": "Sydney",
    "SYDNEY": "Sydney",

    "PHILIP ISLAND TROPHY": "Phillip Island Trophy",
    "PHILLIP ISLAND TROPHY": "Phillip Island Trophy",

    "U.S. OPEN": "Us Open",
    "US OPEN": "Us Open",
}


# ============================================================
# Excluded tournaments
# ============================================================

EXCLUDED_TOURNAMENTS_EXACT = {
    "ATP CUP",
    "UNITED CUP",
    "LAVER CUP",
    "NITTO ATP FINALS",
    "WTA FINALS",
    "NEXT GEN ATP FINALS",
    "GRAMPIANS TROPHY",
    "INTERNATIONAL TENNIS HALL OF FAME",
}

EXCLUDED_TOURNAMENT_PREFIXES = (
    "DAVIS CUP",
    "BILLIE JEAN KING CUP",
    "BNP PARIBAS FED CUP",
)

# ============================================================
# Build normalized lookup from tournament match data
# ============================================================

match_tournament_names_df = (
    player_tournament_df[["tourney_name"]]
    .drop_duplicates()
    .copy()
)

match_tournament_names_df["normalized_tournament"] = (
    match_tournament_names_df["tourney_name"]
    .apply(normalize_tournament_name)
)

# Check that normalization itself does not merge different tournament names
normalization_conflicts_df = (
    match_tournament_names_df
    .groupby("normalized_tournament")
    .agg(
        num_tournament_names=("tourney_name", "nunique"),
        tournament_names=("tourney_name", lambda x: sorted(set(x)))
    )
    .reset_index()
)

normalization_conflicts_df = normalization_conflicts_df[
    normalization_conflicts_df["num_tournament_names"] > 1
]

print("Tournament normalization conflicts:", len(normalization_conflicts_df))

if not normalization_conflicts_df.empty:
    display(normalization_conflicts_df)

match_tournament_normalized_lookup = (
    match_tournament_names_df
    .drop_duplicates(subset=["normalized_tournament"])
    .set_index("normalized_tournament")["tourney_name"]
    .to_dict()
)

# ============================================================
# Special tournament handling
# ============================================================

MELBOURNE_PLAYER_MAPPING = {
    "Grigor Dimitrov": "Melbourne",
    "Jessica Pegula": "Melbourne 2",
    "Simona Halep": "Melbourne 1",
}

ADELAIDE_SPECIAL_DATES = {
    "Adelaide 1": [
        pd.Timestamp("2022-01-03"),
        pd.Timestamp("2023-01-02"),
    ],
    "Adelaide 2": [
        pd.Timestamp("2022-01-10"),
        pd.Timestamp("2023-01-09"),
    ],
}

SPECIAL_YEAR_MAPPING = {
    ("GUADALAJARA OPEN", 2022): "Guadalajara 2",
    ("GUADALAJARA OPEN", 2023): "Guadalajara",

    ("ADELAIDE INTERNATIONAL", 2020): "Adelaide",
    ("ADELAIDE INTERNATIONAL", 2021): "Adelaide",
    ("ADELAIDE INTERNATIONAL", 2024): "Adelaide",
    ("ADELAIDE INTERNATIONAL", 2025): "Adelaide",
    ("ADELAIDE INTERNATIONAL", 2026): "Adelaide",
}

def resolve_special_tournament(tournament, interview_date, player):
    if pd.isna(tournament):
        return None

    tournament_upper = str(tournament).strip().upper()

    if pd.notna(interview_date):
        interview_date = pd.Timestamp(interview_date)

    if tournament_upper == "INTERNATIONAL TENNIS HALL OF FAME":
        return None

    if tournament_upper == "GUADALAJARA OPEN":
        if pd.isna(interview_date):
            return None
        if pd.Timestamp("2022-10-01") <= interview_date <= pd.Timestamp("2022-10-31"):
            return "Guadalajara 2"
        if pd.Timestamp("2023-09-01") <= interview_date <= pd.Timestamp("2023-09-30"):
            return "Guadalajara"
        return None

    if tournament_upper == "MELBOURNE SUMMER SET":
        return MELBOURNE_PLAYER_MAPPING.get(player)

    if tournament_upper == "ADELAIDE INTERNATIONAL":
        if pd.isna(interview_date):
            return None

        if pd.Timestamp("2019-12-20") <= interview_date <= pd.Timestamp("2020-02-01"):
            return "Adelaide"

        if pd.Timestamp("2021-02-01") <= interview_date <= pd.Timestamp("2021-03-10"):
            return "Adelaide"

        if pd.Timestamp("2021-12-20") <= interview_date <= pd.Timestamp("2023-01-20"):
            candidates = []

            for tournament_name, dates in ADELAIDE_SPECIAL_DATES.items():
                for tournament_date in dates:
                    candidates.append((
                        abs((interview_date - tournament_date).days),
                        tournament_name
                    ))

            return min(candidates)[1]

        if pd.Timestamp("2023-12-20") <= interview_date <= pd.Timestamp("2024-01-31"):
            return "Adelaide"

        if pd.Timestamp("2024-12-20") <= interview_date <= pd.Timestamp("2025-01-31"):
            return "Adelaide"

        if pd.Timestamp("2025-12-20") <= interview_date <= pd.Timestamp("2026-01-31"):
            return "Adelaide"

        return None

    return None

# ============================================================
# General tournament resolver
# ============================================================

SPECIAL_TOURNAMENTS = {
    "ADELAIDE INTERNATIONAL",
    "GUADALAJARA OPEN",
    "MELBOURNE SUMMER SET",
    "INTERNATIONAL TENNIS HALL OF FAME",
}

def resolve_tournament_name(tournament, interview_date=None, player=None):
    if pd.isna(tournament):
        return None

    tournament = str(tournament).strip()
    tournament_upper = tournament.upper()

    # 1. Excluded
    if (
        tournament_upper in EXCLUDED_TOURNAMENTS_EXACT
        or tournament_upper.startswith(EXCLUDED_TOURNAMENT_PREFIXES)
    ):
        return None

    # 2. Special cases
    if tournament_upper in SPECIAL_TOURNAMENTS:
        return resolve_special_tournament(
            tournament=tournament,
            interview_date=interview_date,
            player=player
        )

    # 3. Known manual mapping
    if tournament_upper in TOURNAMENT_CANONICAL_MAPPING:
        return TOURNAMENT_CANONICAL_MAPPING[tournament_upper]

    # 4. Simple normalization
    normalized = normalize_tournament_name(tournament)

    if normalized in match_tournament_normalized_lookup:
        return match_tournament_normalized_lookup[normalized]

    return None

# ============================================================
# Map player names
# ============================================================

interviews_df["mapped_player"] = (
    interviews_df["player"]
    .map(PLAYER_NAME_MAPPING)
    .fillna(interviews_df["player"])
)

# ============================================================
# Resolve tournament names in saved interviews
# ============================================================

interviews_df["interview_date"] = pd.to_datetime(
    interviews_df["interview_date"],
    errors="coerce"
)

interviews_df["canonical_tournament"] = interviews_df.apply(
    lambda row: resolve_tournament_name(
        tournament=row["tournament"],
        interview_date=row["interview_date"],
        player=row["mapped_player"]
    ),
    axis=1
)

# ============================================================
# Player-specific mapping for ambiguous cases
# ============================================================

interviews_df["mapping_year"] = interviews_df["interview_date"].dt.year

# December interviews for Adelaide 2023 belong to tournament year 2023
interviews_df.loc[
    interviews_df["canonical_tournament"].isin(
        ["Adelaide 1", "Adelaide 2"]
    )
    & (interviews_df["interview_date"].dt.month == 12),
    "mapping_year"
] = (
    interviews_df.loc[
        interviews_df["canonical_tournament"].isin(
            ["Adelaide 1", "Adelaide 2"]
        )
        & (interviews_df["interview_date"].dt.month == 12),
        "interview_date"
    ].dt.year + 1
)

# Only cases that cannot be solved by tournament + year alone
player_specific_mask = (
    (
        interviews_df["tournament"].str.strip().str.upper()
        == "MELBOURNE SUMMER SET"
    )
    |
    (
        (
            interviews_df["tournament"].str.strip().str.upper()
            == "ADELAIDE INTERNATIONAL"
        )
        & interviews_df["mapping_year"].isin([2022, 2023])
    )
)

special_mapping_df = (
    interviews_df[
        player_specific_mask
        & interviews_df["canonical_tournament"].notna()
    ][
        [
            "mapped_player",
            "tournament",
            "mapping_year",
            "canonical_tournament"
        ]
    ]
    .drop_duplicates()
    .copy()
)

special_mapping_conflicts_df = (
    special_mapping_df
    .groupby(
        ["mapped_player", "tournament", "mapping_year"]
    )
    .agg(
        num_targets=("canonical_tournament", "nunique"),
        targets=(
            "canonical_tournament",
            lambda x: sorted(set(x))
        )
    )
    .reset_index()
)

special_mapping_conflicts_df = special_mapping_conflicts_df[
    special_mapping_conflicts_df["num_targets"] > 1
]

print("Special player-specific mapping conflicts:", len(special_mapping_conflicts_df))

if not special_mapping_conflicts_df.empty:
    display(special_mapping_conflicts_df)
    raise ValueError("Some player+tournament+year keys map to multiple tournaments.")

SPECIAL_TOURNAMENT_MAPPING = {
    (
        row["mapped_player"],
        str(row["tournament"]).strip().upper(),
        int(row["mapping_year"])
    ): row["canonical_tournament"]

    for _, row in special_mapping_df.iterrows()
}

print("Special player-specific tournament mappings:", len(SPECIAL_TOURNAMENT_MAPPING))
print("Special tournament-year mappings:", len(SPECIAL_YEAR_MAPPING))

# ============================================================
# Final checks
# ============================================================

print("\nResolved interview tournament names:")
print(interviews_df["canonical_tournament"].notna().value_counts())

unresolved_interview_tournaments_df = (
    interviews_df[
        interviews_df["canonical_tournament"].isna()
        & ~interviews_df["tournament"].astype("string").str.strip().str.upper().isin(
            EXCLUDED_TOURNAMENTS_EXACT
        )
        & ~interviews_df["tournament"].astype("string").str.strip().str.upper().str.startswith(
            EXCLUDED_TOURNAMENT_PREFIXES
        )
    ][
        [
            "player",
            "mapped_player",
            "tournament",
            "interview_date"
        ]
    ]
    .drop_duplicates()
)

print("Unresolved non-excluded interview rows:", len(unresolved_interview_tournaments_df))
display(unresolved_interview_tournaments_df)

## Dataset Integration
The player-tournament dataset is merged with the interview availability and pre-tournament interview data after applying the validated player and tournament name mappings.

In [ ]:
# ============================================================
# Prepare dates and matching years
# ============================================================

player_tournament_df["tourney_date"] = pd.to_datetime(player_tournament_df["tourney_date"], errors="coerce")
player_tournament_df["tourney_year"] = pd.to_numeric(player_tournament_df["tourney_year"], errors="coerce").astype("Int64")

# Matching year is temporary and used only for merging.
# The original tourney_year is preserved.
player_tournament_df["match_year"] = player_tournament_df["tourney_year"].copy()

# Brisbane tournament starting in December belongs to the next
# tournament matching year.
brisbane_december_mask = (
    player_tournament_df["tourney_name"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("brisbane")
    &
    player_tournament_df["tourney_date"].dt.month.eq(12)
)

player_tournament_df.loc[brisbane_december_mask, "match_year"] = player_tournament_df.loc[brisbane_december_mask, "tourney_year"] + 1

# Status year
status_df["year"] = pd.to_numeric(status_df["year"], errors="coerce").astype("Int64")
status_df["match_year"] = status_df["year"].copy()

# Interview dates
interviews_df["interview_date"] = pd.to_datetime(interviews_df["interview_date"], errors="coerce")

# Calendar year in which the interview was published
interviews_df["interview_year"] = interviews_df["interview_date"].dt.year.astype("Int64")

# Matching year used only during merges
interviews_df["match_year"] = interviews_df["interview_year"].copy()

# Adelaide / Brisbane interviews published in December may belong
# to the following tournament year.
december_next_year_tournaments = {
    "ADELAIDE INTERNATIONAL",
    "BRISBANE INTERNATIONAL",
}

next_year_mask = (
    interviews_df["tournament"]
    .astype("string")
    .str.strip()
    .str.upper()
    .isin(december_next_year_tournaments)
    &
    interviews_df["interview_date"].dt.month.eq(12)
)

interviews_df.loc[next_year_mask, "match_year"] = interviews_df.loc[next_year_mask, "interview_year"] + 1


# ============================================================
# Tournament mapping
# ============================================================

def map_tournament_name(tournament, year=None, player=None):
    if pd.isna(tournament):
        return None

    tournament_upper = str(tournament).strip().upper()

    # 1. Excluded tournaments
    if (tournament_upper in EXCLUDED_TOURNAMENTS_EXACT
        or tournament_upper.startswith(EXCLUDED_TOURNAMENT_PREFIXES)):
        return None

    # 2. Tournament + year mapping
    if year is not None and pd.notna(year):
        year_key = (tournament_upper, int(year))

        if year_key in SPECIAL_YEAR_MAPPING:
            return SPECIAL_YEAR_MAPPING[year_key]

    # 3. Player + tournament + year mapping
    if (player is not None
        and pd.notna(player)
        and year is not None
        and pd.notna(year)):
        player_key = (player, tournament_upper, int(year))

        if player_key in SPECIAL_TOURNAMENT_MAPPING:
            return SPECIAL_TOURNAMENT_MAPPING[player_key]

    # 4. Known tournament-name mapping
    if tournament_upper in TOURNAMENT_CANONICAL_MAPPING:
        return TOURNAMENT_CANONICAL_MAPPING[tournament_upper]

    # 5. Simple normalization
    normalized = normalize_tournament_name(tournament)

    if normalized in match_tournament_normalized_lookup:
        return match_tournament_normalized_lookup[normalized]

    # Keep the original name if no explicit mapping is needed
    return tournament


def prepare_match_text(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )


# ============================================================
# Map player names
# ============================================================

player_tournament_df["match_player"] = prepare_match_text(player_tournament_df["player_name"])

status_df["mapped_player"] = (
    status_df["player"]
    .map(PLAYER_NAME_MAPPING)
    .fillna(status_df["player"])
)

interviews_df["mapped_player"] = (
    interviews_df["player"]
    .map(PLAYER_NAME_MAPPING)
    .fillna(interviews_df["player"])
)


# ============================================================
# Adjust status matching year using saved interviews
# ============================================================

status_df["_tournament_upper"] = (
    status_df["tournament"]
    .astype("string")
    .str.strip()
    .str.upper()
)

interviews_df["_tournament_upper"] = (
    interviews_df["tournament"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Interviews whose calendar year differs from their matching year
shifted_interview_years_df = (
    interviews_df[
        interviews_df["match_year"]
        != interviews_df["interview_year"]
    ][
        [
            "mapped_player",
            "_tournament_upper",
            "interview_year",
            "match_year"
        ]
    ]
    .drop_duplicates()
)

# Check that one player+tournament+calendar-year does not map
# to multiple matching years.
shift_conflicts_df = (
    shifted_interview_years_df
    .groupby(
        [
            "mapped_player",
            "_tournament_upper",
            "interview_year"
        ]
    )["match_year"]
    .nunique()
)

if (shift_conflicts_df > 1).any():
    raise ValueError("Some status year mappings have multiple possible match years.")

shifted_status_year_lookup = {
    (
        row["mapped_player"],
        row["_tournament_upper"],
        int(row["interview_year"])
    ): int(row["match_year"])

    for _, row in shifted_interview_years_df.iterrows()
}


def get_status_match_year(row):
    if pd.isna(row["year"]):
        return row["match_year"]

    key = (
        row["mapped_player"],
        row["_tournament_upper"],
        int(row["year"])
    )

    return shifted_status_year_lookup.get(
        key,
        row["match_year"]
    )


status_df["match_year"] = (
    status_df
    .apply(
        get_status_match_year,
        axis=1
    )
    .astype("Int64")
)


# ============================================================
# Validate shifted status years
# ============================================================

shifted_status_rows_df = status_df[
    status_df["year"] != status_df["match_year"]
].copy()

print("Status rows shifted to another matching year:", len(shifted_status_rows_df))

if not shifted_status_rows_df.empty:
    display(
        shifted_status_rows_df[
            [
                "player",
                "tournament",
                "year",
                "match_year",
                "has_any_interview",
                "has_pre_match_interview"
            ]
        ].sort_values(
            [
                "tournament",
                "match_year",
                "player"
            ]
        )
    )

    print("Tournaments with shifted status year:", sorted(shifted_status_rows_df["tournament"].dropna().unique()))


# ============================================================
# Map tournament names
# ============================================================

status_df["mapped_tournament"] = status_df.apply(
    lambda row: map_tournament_name(
        tournament=row["tournament"],
        year=row["match_year"],
        player=row["mapped_player"]
    ),
    axis=1
)

interviews_df["mapped_tournament"] = interviews_df.apply(
    lambda row: map_tournament_name(
        tournament=row["tournament"],
        year=row["match_year"],
        player=row["mapped_player"]
    ),
    axis=1
)

player_tournament_df["mapped_tournament"] = (
    player_tournament_df.apply(
        lambda row: map_tournament_name(
            tournament=row["tourney_name"],
            year=row["match_year"],
            player=row["player_name"]
        ),
        axis=1
    )
)


# ============================================================
# Check unresolved tournament mappings
# ============================================================

def is_excluded_tournament(tournament):
    if pd.isna(tournament):
        return False

    tournament_upper = str(tournament).strip().upper()

    return (
        tournament_upper in EXCLUDED_TOURNAMENTS_EXACT
        or tournament_upper.startswith(
            EXCLUDED_TOURNAMENT_PREFIXES
        )
    )


# Interviews
unresolved_interviews_mask = interviews_df["mapped_tournament"].isna()

excluded_unresolved_interviews_mask = (
    unresolved_interviews_mask
    & interviews_df["tournament"]
    .apply(is_excluded_tournament)
)

unresolved_interviews_to_investigate_mask = (
    unresolved_interviews_mask
    & ~interviews_df["tournament"]
    .apply(is_excluded_tournament)
)

print("\nInterview tournament mappings:")
print("Unresolved:", int(unresolved_interviews_mask.sum()))
print("  Excluded tournaments:", int(excluded_unresolved_interviews_mask.sum()))
print("  Require investigation:", int(unresolved_interviews_to_investigate_mask.sum()))

# Status
unresolved_status_mask = status_df["mapped_tournament"].isna()

excluded_unresolved_status_mask = (
    unresolved_status_mask
    & status_df["tournament"]
    .apply(is_excluded_tournament)
)

unresolved_status_to_investigate_mask = (
    unresolved_status_mask
    & ~status_df["tournament"]
    .apply(is_excluded_tournament)
)

print("\nStatus tournament mappings:")
print("Unresolved:", int(unresolved_status_mask.sum()))
print("  Excluded tournaments:", int(excluded_unresolved_status_mask.sum()))
print("  Require investigation:", int(unresolved_status_to_investigate_mask.sum()))
print("Unresolved match tournaments:", player_tournament_df["mapped_tournament"].isna().sum())


# ============================================================
# Prepare merge keys
# ============================================================

player_tournament_df["match_tournament"] = prepare_match_text(player_tournament_df["mapped_tournament"])
status_df["match_player"] = prepare_match_text(status_df["mapped_player"])
status_df["match_tournament"] = prepare_match_text(status_df["mapped_tournament"])
interviews_df["match_player"] = prepare_match_text(interviews_df["mapped_player"])
interviews_df["match_tournament"] = prepare_match_text(interviews_df["mapped_tournament"])

MERGE_KEY = [
    "match_player",
    "match_tournament",
    "match_year"
]


# ============================================================
# Convert status columns to boolean
# ============================================================

def parse_bool(value):
    if pd.isna(value):
        return False

    if isinstance(value, bool):
        return value

    return (
        str(value)
        .strip()
        .lower()
        in {"true", "1", "yes"}
    )


status_df["has_any_interview"] = (
    status_df["has_any_interview"]
    .apply(parse_bool)
)

status_df["has_pre_match_interview"] = (
    status_df["has_pre_match_interview"]
    .apply(parse_bool)
)


# ============================================================
# Check invalid status rows
# ============================================================

invalid_status_rows_df = status_df[
    status_df["has_pre_match_interview"]
    & ~status_df["has_any_interview"]
].copy()

print(
    "\nInvalid status rows:",
    len(invalid_status_rows_df)
)

if not invalid_status_rows_df.empty:
    display(
        invalid_status_rows_df[
            [
                "player",
                "match_year",
                "tournament",
                "has_any_interview",
                "has_pre_match_interview"
            ]
        ]
    )


# ============================================================
# Check duplicate status keys
# ============================================================

duplicate_status_keys_df = (
    status_df
    .groupby(MERGE_KEY)
    .size()
    .reset_index(name="num_rows")
)

duplicate_status_keys_df = (
    duplicate_status_keys_df[
        duplicate_status_keys_df["num_rows"] > 1
    ]
)

print("Duplicate status keys after player and tournament mapping:", len(duplicate_status_keys_df))

if not duplicate_status_keys_df.empty:
    duplicate_status_rows_df = (
        status_df
        .merge(
            duplicate_status_keys_df[MERGE_KEY],
            on=MERGE_KEY,
            how="inner"
        )
        .sort_values(MERGE_KEY)
    )

    display(
        duplicate_status_rows_df[
            [
                "player",
                "mapped_player",
                "tournament",
                "match_year",
                "has_any_interview",
                "has_pre_match_interview"
            ]
        ]
    )


# Combine duplicate status rows using OR
status_lookup_df = (
    status_df
    .groupby(
        MERGE_KEY,
        as_index=False,
        dropna=False
    )
    .agg(
        has_any_interview=(
            "has_any_interview",
            "max"
        ),
        has_pre_match_interview=(
            "has_pre_match_interview",
            "max"
        ),
        status_player_original=(
            "player",
            lambda values: " | ".join(
                sorted(set(values.dropna()))
            )
        ),
        status_tournament_original=(
            "tournament",
            "first"
        )
    )
)

if status_lookup_df.duplicated(MERGE_KEY).any():
    raise ValueError("Status lookup still contains duplicate keys.")


# ============================================================
# Check duplicate interview keys
# ============================================================

duplicate_interview_keys_df = (
    interviews_df
    .groupby(MERGE_KEY)
    .size()
    .reset_index(name="num_rows")
)

duplicate_interview_keys_df = duplicate_interview_keys_df[duplicate_interview_keys_df["num_rows"] > 1]


print("Duplicate saved pre-tournament keys after player and tournament mapping:", len(duplicate_interview_keys_df))

if not duplicate_interview_keys_df.empty:
    duplicate_interview_rows_df = (
        interviews_df
        .merge(
            duplicate_interview_keys_df[
                MERGE_KEY
            ],
            on=MERGE_KEY,
            how="inner"
        )
        .sort_values(
            MERGE_KEY + ["interview_date"]
        )
    )

    display(
        duplicate_interview_rows_df[
            [
                "player",
                "mapped_player",
                "tournament",
                "match_year",
                "interview_date",
                "url"
            ]
        ]
    )


# ============================================================
# Keep earliest saved pre-tournament interview
# ============================================================

interviews_df["_original_row_order"] = range(len(interviews_df))

first_interviews_lookup_df = (
    interviews_df
    .sort_values(
        MERGE_KEY
        + [
            "interview_date",
            "_original_row_order"
        ],
        na_position="last"
    )
    .drop_duplicates(
        subset=MERGE_KEY,
        keep="first"
    )
    .rename(
        columns={
            "player":
                "first_interview_player_original",
            "tournament":
                "first_interview_tournament_original",
            "interview_date":
                "first_pre_match_interview_date",
            "url":
                "first_pre_match_interview_url",
            "qa_json":
                "first_pre_match_interview_qa_json"
        }
    )
)

first_interviews_lookup_df = (
    first_interviews_lookup_df[
        MERGE_KEY
        + [
            "first_interview_player_original",
            "first_interview_tournament_original",
            "first_pre_match_interview_date",
            "first_pre_match_interview_url",
            "first_pre_match_interview_qa_json"
        ]
    ]
    .copy()
)

if first_interviews_lookup_df.duplicated(MERGE_KEY).any():
    raise ValueError("First-interview lookup still contains duplicate keys.")

# ============================================================
# Show final result for duplicate status keys
# ============================================================

final_duplicate_results_df = (
    duplicate_status_keys_df
    .merge(
        status_lookup_df,
        on=MERGE_KEY,
        how="left"
    )
    .merge(
        first_interviews_lookup_df,
        on=MERGE_KEY,
        how="left"
    )
)

if not final_duplicate_results_df.empty:
    display(
        final_duplicate_results_df[
            [
                "match_player",
                "match_tournament",
                "match_year",
                "has_any_interview",
                "has_pre_match_interview",
                "first_interview_player_original",
                "first_pre_match_interview_date",
                "first_pre_match_interview_url"
            ]
        ]
    )


# ============================================================
# Merge status information
# ============================================================

expected_rows = len(player_tournament_df)

player_tournament_with_interviews_df = (
    player_tournament_df.merge(
        status_lookup_df,
        on=MERGE_KEY,
        how="left",
        validate="many_to_one"
    )
)

if (len(player_tournament_with_interviews_df) != expected_rows):
    raise ValueError("Status merge changed the number of rows.")

# ============================================================
# Merge first pre-tournament interview content
# ============================================================

player_tournament_with_interviews_df = (
    player_tournament_with_interviews_df.merge(
        first_interviews_lookup_df,
        on=MERGE_KEY,
        how="left",
        validate="many_to_one"
    )
)

if (len(player_tournament_with_interviews_df) != expected_rows):
    raise ValueError("Interview merge changed the number of rows.")

# ============================================================
# Fill missing boolean values
# ============================================================

player_tournament_with_interviews_df["has_any_interview"] = (
    player_tournament_with_interviews_df["has_any_interview"].astype("boolean").fillna(False))

player_tournament_with_interviews_df["has_pre_match_interview"] = (
    player_tournament_with_interviews_df["has_pre_match_interview"].astype("boolean").fillna(False))

player_tournament_with_interviews_df["has_saved_first_pre_match"] = (
    player_tournament_with_interviews_df["first_pre_match_interview_qa_json"].notna())


# ============================================================
# Validate interview dates
# ============================================================

player_tournament_with_interviews_df["first_interview_days_from_tournament_start"] = (
    player_tournament_with_interviews_df["first_pre_match_interview_date"]
    - player_tournament_with_interviews_df["tourney_date"]).dt.days

suspicious_interview_dates_df = (player_tournament_with_interviews_df[player_tournament_with_interviews_df["has_saved_first_pre_match"] &
        ~player_tournament_with_interviews_df["first_interview_days_from_tournament_start"].between(-7, 21)].copy())


# ============================================================
# Check consistency between status and saved interview content
# ============================================================

inconsistent_pre_match_status_df = (player_tournament_with_interviews_df[
        player_tournament_with_interviews_df["has_pre_match_interview"].astype(bool)
        != player_tournament_with_interviews_df["has_saved_first_pre_match"]].copy())


# ============================================================
# Find status rows not matched to tournament data
# ============================================================

player_tournament_keys_df = player_tournament_df[MERGE_KEY].drop_duplicates()

unmatched_status_df = (
    status_lookup_df
    .merge(
        player_tournament_keys_df,
        on=MERGE_KEY,
        how="left",
        indicator=True
    )
)

unmatched_status_df = unmatched_status_df[unmatched_status_df["_merge"] == "left_only"].drop(columns="_merge")

# ============================================================
# Find saved interviews not matched to tournament data
# ============================================================

unmatched_first_interviews_df = (
    first_interviews_lookup_df
    .merge(
        player_tournament_keys_df,
        on=MERGE_KEY,
        how="left",
        indicator=True
    )
)

unmatched_first_interviews_df = (
    unmatched_first_interviews_df[
        unmatched_first_interviews_df[
            "_merge"
        ] == "left_only"
    ]
    .drop(columns="_merge")
)


# ============================================================
# Check tournament names changed by mapping
# ============================================================

tournament_mapping_changes_df = (
    status_df[
        status_df["tournament"]
        .astype("string")
        .str.strip()
        !=
        status_df["mapped_tournament"]
        .astype("string")
        .str.strip()
    ][
        [
            "tournament",
            "mapped_tournament"
        ]
    ]
    .drop_duplicates()
)

print("\nTournament names changed by mapping:", len(tournament_mapping_changes_df))

if not tournament_mapping_changes_df.empty:
    display(tournament_mapping_changes_df)

# ============================================================
# Count unmatched rows belonging to excluded tournaments
# ============================================================

excluded_status_rows = unmatched_status_df["status_tournament_original"].astype("string").str.strip().str.upper().apply(is_excluded_tournament)

excluded_saved_rows = unmatched_first_interviews_df["first_interview_tournament_original"].astype("string").str.strip().str.upper().apply(is_excluded_tournament)

# ============================================================
# Summary
# ============================================================

print("\nFinal player-tournament rows:", len(player_tournament_with_interviews_df))
print("Rows where the player had any interview:", int(player_tournament_with_interviews_df["has_any_interview"].sum()))
print("Rows where the player had a pre-tournament interview:", int(player_tournament_with_interviews_df["has_pre_match_interview"].sum()))
print("Rows with saved first pre-tournament content:", int(player_tournament_with_interviews_df["has_saved_first_pre_match"].sum()))

print("Unmatched status rows:", len(unmatched_status_df),
    f"(excluded: {excluded_status_rows.sum()}, "
    f"non-excluded: {(~excluded_status_rows).sum()})"
)

print("Unmatched saved interview rows:", len(unmatched_first_interviews_df),
    f"(excluded: {excluded_saved_rows.sum()}, "
    f"non-excluded: {(~excluded_saved_rows).sum()})"
)

print("\nNon-excluded unmatched status rows:")

display(
    unmatched_status_df[
        ~excluded_status_rows
    ][
        [
            "status_player_original",
            "match_year",
            "status_tournament_original",
            "has_any_interview",
            "has_pre_match_interview"
        ]
    ].head(50)
)

print("\nNon-excluded unmatched saved interviews:")

display(
    unmatched_first_interviews_df[
        ~excluded_saved_rows
    ][
        [
            "first_interview_player_original",
            "match_year",
            "first_interview_tournament_original",
            "first_pre_match_interview_date"
        ]
    ].head(50)
)

print("Status-content inconsistencies:", len(inconsistent_pre_match_status_df))
print("Suspicious interview dates:", len(suspicious_interview_dates_df))

# ============================================================
# Display remaining validation issues
# ============================================================

print("\nStatus-content inconsistencies:")

display(
    inconsistent_pre_match_status_df[
        [
            "player_name",
            "tourney_year",
            "tourney_name",
            "has_any_interview",
            "has_pre_match_interview",
            "has_saved_first_pre_match",
            "first_pre_match_interview_date"
        ]
    ].head(50)
)

print("\nSuspicious interview dates:")

display(
    suspicious_interview_dates_df[
        [
            "player_name",
            "tourney_year",
            "tourney_name",
            "tourney_date",
            "first_pre_match_interview_date",
            "first_interview_days_from_tournament_start"
        ]
    ].head(50)
)


# ============================================================
# Remove temporary merge columns
# ============================================================

player_tournament_with_interviews_df = (
    player_tournament_with_interviews_df.drop(
        columns=[
            "match_year",
            "match_player",
            "match_tournament",
            "mapped_tournament",
            "status_player_original",
            "status_tournament_original",
            "first_interview_player_original",
            "first_interview_tournament_original",
            "has_saved_first_pre_match",
            "first_interview_days_from_tournament_start",
        ],
        errors="ignore"
    )
)

status_df = status_df.drop(
    columns=[
        "match_year",
        "match_player",
        "match_tournament",
        "mapped_player",
        "mapped_tournament",
        "_tournament_upper",
    ],
    errors="ignore"
)

interviews_df = interviews_df.drop(
    columns=[
        "match_year",
        "match_player",
        "match_tournament",
        "mapped_player",
        "mapped_tournament",
        "_tournament_upper",
        "_original_row_order",
    ],
    errors="ignore"
)



In [ ]:
# ============================================================
# Save merged dataset
# ============================================================

player_tournament_with_interviews_df.to_csv(MERGED_DATASET_FILE, index=False)

print("Saved merged dataset to:")
print(MERGED_DATASET_FILE)

## Feature Engineering

In [ ]:
# ============================================================
# Check missing categorical values before encoding
# ============================================================

categorical_columns = ["tour", "season", "surface"]

print("Missing values before encoding:")

for col in categorical_columns:
    missing_count = player_tournament_with_interviews_df[col].isna().sum()

    print(f"\n{col}: {missing_count} missing values")

    if missing_count > 0:
        display(
            player_tournament_with_interviews_df.loc[
                player_tournament_with_interviews_df[col].isna(),
                [
                    "player_name",
                    "tourney_name",
                    "tourney_year",
                    col,
                ]
            ].drop_duplicates()
        )

print("\nValues missing data would receive with the current encoding:")
print("Missing tour    -> is_male = 0 (would look like WTA)")
print("Missing season  -> 0, 0, 0 (would look like Fall)")
print("Missing surface -> 0, 0 (would look like Grass)")

# ============================================================
# Encode categorical features
# ============================================================

player_tournament_interview_features_df = player_tournament_with_interviews_df.copy()

# ------------------------------------------------------------
# Tour
# 1 = Male / ATP
# 0 = Female / WTA
# ------------------------------------------------------------

player_tournament_interview_features_df["is_male"] = (
    player_tournament_interview_features_df["tour"]
    .eq("ATP")
    .astype(int)
)


# ------------------------------------------------------------
# Season
# Fall is the reference category:
# Fall   -> 0, 0, 0
# Spring -> 1, 0, 0
# Summer -> 0, 1, 0
# Winter -> 0, 0, 1
# ------------------------------------------------------------

player_tournament_interview_features_df["is_spring"] = (
    player_tournament_interview_features_df["season"]
    .eq("Spring")
    .astype(int)
)

player_tournament_interview_features_df["is_summer"] = (
    player_tournament_interview_features_df["season"]
    .eq("Summer")
    .astype(int)
)

player_tournament_interview_features_df["is_winter"] = (
    player_tournament_interview_features_df["season"]
    .eq("Winter")
    .astype(int)
)


# ------------------------------------------------------------
# Surface
# Grass is the reference category:
# Grass -> 0, 0
# Hard  -> 1, 0
# Clay  -> 0, 1
# ------------------------------------------------------------

player_tournament_interview_features_df["is_hard_surface"] = (
    player_tournament_interview_features_df["surface"]
    .eq("Hard")
    .astype(int)
)

player_tournament_interview_features_df["is_clay_surface"] = (
    player_tournament_interview_features_df["surface"]
    .eq("Clay")
    .astype(int)
)


# ============================================================
# Validation
# ============================================================

# Check that original categorical values are expected
unexpected_tours = set(
    player_tournament_interview_features_df["tour"]
    .dropna()
    .unique()
) - {"ATP", "WTA"}

unexpected_seasons = set(
    player_tournament_interview_features_df["season"]
    .dropna()
    .unique()
) - {"Spring", "Summer", "Winter", "Fall"}

unexpected_surfaces = set(
    player_tournament_interview_features_df["surface"]
    .dropna()
    .unique()
) - {"Hard", "Clay", "Grass"}

print("Unexpected tour values:", unexpected_tours)
print("Unexpected season values:", unexpected_seasons)
print("Unexpected surface values:", unexpected_surfaces)

# Check encoded distributions
print("\nis_male:")
print(player_tournament_interview_features_df["is_male"].value_counts().sort_index())

print("\nSeason encoding:")
print(player_tournament_interview_features_df[["is_spring", "is_summer", "is_winter"]].value_counts().sort_index())

print("\nSurface encoding:")
print(player_tournament_interview_features_df[["is_hard_surface", "is_clay_surface"]].value_counts().sort_index())

# ============================================================
# Remove original categorical columns
# ============================================================

player_tournament_interview_features_df = (
    player_tournament_interview_features_df.drop(
        columns=[
            "tour",
            "season",
            "surface",
        ]
    )
)

In [ ]:
# ============================================================
# Tournament level feature
# Higher value = higher tournament level
# ============================================================

# 0 = ITF / WTA 125
# 1 = WTA 250
# 2 = ATP Tour / WTA 500
# 3 = Masters 1000 / WTA 1000
# 4 = Grand Slam

missing_level = player_tournament_interview_features_df["tourney_level"].isna()

print("Missing tourney_level:", missing_level.sum())

if missing_level.any():
    display(
        player_tournament_interview_features_df.loc[
            missing_level,
            [
                "tourney_name",
                "tourney_year",
                "tourney_level",
                "is_male",
            ]
        ]
        .drop_duplicates()
        .sort_values(["is_male", "tourney_year", "tourney_name"])
    )


# ------------------------------------------------------------
# ATP levels
# ------------------------------------------------------------

ATP_LEVEL_SCORE = {
    "A": 2,   # ATP Tour event
    "M": 3,   # Masters 1000
    "G": 4,   # Grand Slam
}


# ------------------------------------------------------------
# WTA levels
# ------------------------------------------------------------

WTA_LEVEL_SCORE = {
    "35+H": 0,   # ITF level
    "50+H": 0,   # ITF level
    "I": 1,      # WTA 250 level
    "P": 2,      # WTA 500 level
    "PM": 3,     # WTA 1000 level
    "G": 4,      # Grand Slam
}


# ------------------------------------------------------------
# WTA tournaments coded as "W"
# ------------------------------------------------------------

WTA_W_LEVEL_SCORE = {
    (2021, "Bad Homburg"): 1,       # WTA 250
    (2021, "Belgrade"): 1,          # WTA 250
    (2021, "Charleston 2"): 1,      # WTA 250
    (2021, "Chicago 1"): 1,         # WTA 250
    (2021, "Cleveland"): 1,         # WTA 250
    (2021, "Eastbourne"): 1,        # WTA 250
    (2021, "Parma"): 1,             # WTA 250
    (2021, "San Jose"): 2,          # WTA 500
    (2021, "Strasbourg"): 1,        # WTA 250
    (2024, "Buenos Aires 125"): 0,  # WTA 125
}


# ------------------------------------------------------------
# Create tournament level score
# ------------------------------------------------------------

def get_tournament_level_score(row):
    level = row["tourney_level"]

    # ATP
    if row["is_male"] == 1:
        return ATP_LEVEL_SCORE.get(level)

    # WTA with explicit level code
    if level in WTA_LEVEL_SCORE:
        return WTA_LEVEL_SCORE[level]

    # WTA tournaments coded as "W"
    if level == "W":
        key = int(row["tourney_year"]),row["tourney_name"]
        return WTA_W_LEVEL_SCORE.get(key)

    return None


player_tournament_interview_features_df["tournament_level_score"] = (
    player_tournament_interview_features_df.apply(
        get_tournament_level_score,
        axis=1
    )
)


# ============================================================
# Validation
# ============================================================

unresolved_level_df = (
    player_tournament_interview_features_df[
        player_tournament_interview_features_df[
            "tournament_level_score"
        ].isna()
    ][
        [
            "tourney_year",
            "tourney_name",
            "tourney_level",
            "is_male",
        ]
    ]
    .drop_duplicates().sort_values(["tourney_level", "tourney_year", "tourney_name"])
)

print("Unresolved tournament levels:", len(unresolved_level_df))

if not unresolved_level_df.empty:
    display(unresolved_level_df)

print("\nTournament level score distribution:")
print(player_tournament_interview_features_df["tournament_level_score"].value_counts().sort_index())

player_tournament_interview_features_df = player_tournament_interview_features_df.drop(columns=["tourney_level"])

In [ ]:
# ============================================================
# Remove unused columns and irrelevant missing-age rows
# ============================================================

player_tournament_interview_features_df = (
    player_tournament_interview_features_df
    # Players with missing age have no pre-tournament interviews anywhere
    .dropna(subset=["player_age"])
    # Remove features that will not be used
    .drop(
        columns=[
            "player_entry",
            "player_seed",
            "player_height",
            "player_rank_points",
        ],
        errors="ignore",
    )
    .copy()
)

# Do NOT remove missing player_rank rows yet.
# They are needed to preserve the true 3 previous tournaments.

print("Rows:", len(player_tournament_interview_features_df))
print("Missing player_age:", player_tournament_interview_features_df["player_age"].isna().sum())
print("Missing player_rank:", player_tournament_interview_features_df["player_rank"].isna().sum())

In [ ]:
# ============================================================
# Save merged dataset with features
# ============================================================

player_tournament_interview_features_df.to_csv(FEATURES_DATASET_FILE, index=False)

print("Saved feature dataset to:")
print(FEATURES_DATASET_FILE)

## Previous Tournament History

For each player, tournaments are ordered chronologically. Features from the three immediately preceding tournaments are attached to the current tournament to represent recent performance and context without using future information.

### Missing Ranking Completion

Missing player rankings are completed where sufficient information is available. Records for which a ranking cannot be recovered are retained rather than removed.

In [ ]:
# ============================================================
# Handle missing ranks, calculate opponent difficulty,
# and add features from the 3 previous tournaments
# ============================================================

df = player_tournament_interview_features_df.copy()
df["tourney_date"] = pd.to_datetime(df["tourney_date"], errors="coerce")

# ============================================================
# 1. PLAYER RANK
# ============================================================
#
# player_rank_available:
#   1 = rank is known
#   0 = rank is missing
#
# Missing rank: player_rank = -1
#
# Important: Keep missing-rank rows so previous-tournament history remains complete
# ============================================================

df["player_rank"] = pd.to_numeric(df["player_rank"], errors="coerce")

# A real rank must be positive.
# NaN and the -1 sentinel both mean unavailable.
df["player_rank_available"] = (df["player_rank"] > 0).astype(int)

df.loc[df["player_rank_available"] == 0, "player_rank"] = -1


# ============================================================
# 2. VALIDATE PLAYER RANK ENCODING
# ============================================================

if df["player_rank"].isna().any():
    raise ValueError("player_rank still contains missing values.")

if not set(df["player_rank_available"].unique()).issubset({0, 1}):
    raise ValueError("player_rank_available contains values other than 0/1.")

# Missing ranks must be represented by -1
invalid_missing_rank = df[(df["player_rank_available"] == 0) & (df["player_rank"] != -1)]

if not invalid_missing_rank.empty:
    raise ValueError("Some unavailable ranks are not encoded as -1.")

# Available ranks must be positive
invalid_available_rank = df[(df["player_rank_available"] == 1) & (df["player_rank"] <= 0)]

if not invalid_available_rank.empty:
    raise ValueError("Some available player ranks are not positive.")

print("Player rank:")
print(df["player_rank_available"].value_counts().sort_index())
print("Rows with unavailable rank:", (df["player_rank_available"] == 0).sum())

# ============================================================
# 3. RANK -> OPPONENT STRENGTH
# ============================================================
#
# Smaller ranking number = stronger player.
#
# We convert ranking into a score where:
# higher score = stronger opponent.
#
# IMPORTANT:
# MAX_RANK is calculated using only real/available ranks.
# The -1 sentinel is never used in this calculation.
# ============================================================

observed_ranks = df.loc[df["player_rank_available"] == 1, "player_rank"]

if observed_ranks.empty:
    raise ValueError("No observed player ranks are available.")

MAX_RANK = int(observed_ranks.max())

if MAX_RANK <= 1:
    raise ValueError("MAX_RANK must be greater than 1.")

def rank_to_strength(rank):
    """
    Convert ATP/WTA rank to a [0, 1] strength score.
    Higher value means a stronger opponent.
    """
    strength = 1 - np.log(rank) / np.log(MAX_RANK)
    return float(np.clip(strength, 0, 1))

print("Maximum observed rank used for strength scaling:", MAX_RANK)


# ============================================================
# 4. PLAYER-TOURNAMENT RANK LOOKUP
# ============================================================

rank_lookup_df = (
    df[
        [
            "player_key",
            "tournament_key",
            "player_rank",
            "player_rank_available",
        ]
    ]
    .drop_duplicates(
        subset=[
            "player_key",
            "tournament_key",
        ]
    )
    .copy()
)

# Verify one rank record per player-tournament
duplicate_rank_keys = (
    rank_lookup_df
    .duplicated(
        subset=[
            "player_key",
            "tournament_key",
        ]
    )
    .sum()
)

if duplicate_rank_keys != 0:
    raise ValueError("Duplicate player-tournament keys in rank lookup.")


# ============================================================
# 5. CREATE PLAYER-OPPONENT DATA
# ============================================================
# Each match becomes two rows:
# winner -> loser
# loser  -> winner
# Compute opponent difficulty for each player in every tournament.
# ============================================================

winner_side_df = (
    all_matches_df_clean[
        [
            "tournament_key",
            "winner_player_key",
            "loser_player_key",
        ]
    ]
    .rename(
        columns={
            "winner_player_key": "player_key",
            "loser_player_key": "opponent_player_key",
        }
    )
)

loser_side_df = (
    all_matches_df_clean[
        [
            "tournament_key",
            "winner_player_key",
            "loser_player_key",
        ]
    ]
    .rename(
        columns={
            "loser_player_key": "player_key",
            "winner_player_key": "opponent_player_key",
        }
    )
)

player_opponent_df = pd.concat(
    [
        winner_side_df,
        loser_side_df,
    ],
    ignore_index=True
)


# ============================================================
# 6. ADD OPPONENT RANK
# ============================================================

opponent_rank_lookup_df = (
    rank_lookup_df.rename(
        columns={
            "player_key": "opponent_player_key",
            "player_rank": "opponent_rank",
            "player_rank_available":
                "opponent_rank_available",
        }
    )
)

player_opponent_df = player_opponent_df.merge(
    opponent_rank_lookup_df,
    on=[
        "opponent_player_key",
        "tournament_key",
    ],
    how="left",
    validate="many_to_one",
)

# Opponent not found in lookup -> rank unavailable
player_opponent_df["opponent_rank_available"] = player_opponent_df["opponent_rank_available"].fillna(0).astype(int)

# ============================================================
# 7. CALCULATE STRENGTH ONLY FOR RANKED OPPONENTS
# ============================================================

player_opponent_df["opponent_strength_score"] = np.nan
available_opponent_rank = player_opponent_df["opponent_rank_available"] == 1
player_opponent_df.loc[available_opponent_rank, "opponent_strength_score"] = player_opponent_df.loc[available_opponent_rank, "opponent_rank"].apply(rank_to_strength)


# ============================================================
# 8. AGGREGATE OPPONENT DIFFICULTY PER TOURNAMENT
# ============================================================
#
# score:
#   mean strength of opponents whose rank is known
#
# confidence:
#   number of opponents with known rank
#   -----------------------------------
#   total number of opponents played
#
# Example:
#
# 4 opponents, ranks known for 3:
# confidence = 3 / 4 = 0.75
#
# If no opponent has a known rank:
# score = -1
# confidence = 0
# ============================================================

def aggregate_opponent_difficulty(group):
    num_opponents = len(group)
    ranked_opponents = group[group["opponent_rank_available"] == 1]
    num_ranked_opponents = len(ranked_opponents)
    confidence = num_ranked_opponents / num_opponents

    if num_ranked_opponents == 0:
        score = -1.0
    else:
        score = ranked_opponents["opponent_strength_score"].mean()

    return pd.Series({
        "opponent_difficulty_score":
            score,
        "opponent_difficulty_confidence":
            confidence,
        "num_opponents":
            num_opponents,
        "num_ranked_opponents":
            num_ranked_opponents,
    })


opponent_difficulty_df = (
    player_opponent_df
    .groupby(
        [
            "player_key",
            "tournament_key",
        ]
    )
    .apply(
        aggregate_opponent_difficulty,
        include_groups=False
    )
    .reset_index()
)


# ============================================================
# 9. MERGE OPPONENT DIFFICULTY INTO TOURNAMENT DATA
# ============================================================

rows_before_merge = len(df)

df = df.merge(
    opponent_difficulty_df,
    on=[
        "player_key",
        "tournament_key",
    ],
    how="left",
    validate="one_to_one",
)

if len(df) != rows_before_merge:
    raise ValueError("Opponent difficulty merge changed row count.")

# Handle player-tournaments with no opponent information
df["opponent_difficulty_score"] = df["opponent_difficulty_score"].fillna(-1)
df["opponent_difficulty_confidence"] = df["opponent_difficulty_confidence"].fillna(0)
df["num_opponents"] = df["num_opponents"].fillna(0).astype(int)
df["num_ranked_opponents"] = df["num_ranked_opponents"].fillna(0).astype(int)

# ============================================================
# 10. VALIDATE OPPONENT DIFFICULTY
# ============================================================

if not df["opponent_difficulty_confidence"].between(0, 1).all():
    raise ValueError("opponent_difficulty_confidence must be between 0 and 1.")

# confidence = 0 <-> no ranked opponents
invalid_zero_confidence = df[(df["opponent_difficulty_confidence"] == 0) & (df["opponent_difficulty_score"] != -1)]

if not invalid_zero_confidence.empty:
    raise ValueError("Rows with confidence=0 must have difficulty score=-1.")

# If confidence > 0, score must be valid
invalid_positive_confidence = df[(df["opponent_difficulty_confidence"] > 0) & (df["opponent_difficulty_score"].isna() | (df["opponent_difficulty_score"] < 0))]

if not invalid_positive_confidence.empty:
    raise ValueError("Rows with opponent data contain an invalid difficulty score.")

print("\nOpponent difficulty:")
print(df["opponent_difficulty_confidence"].describe())
print("Player-tournaments with no ranked opponents:", (df["opponent_difficulty_confidence"] == 0).sum())

# ============================================================
# 11. SORT TOURNAMENTS CHRONOLOGICALLY
# ============================================================

df = (
    df
    .sort_values(
        [
            "player_key",
            "tourney_date",
            "tournament_key",
        ]
    )
    .copy()
)

# Number of tournaments before the current tournament
df["num_previous_tournaments"] = df.groupby("player_key").cumcount()

# ============================================================
# 12. FEATURES TO TAKE FROM THE 3 PREVIOUS TOURNAMENTS
# ============================================================
#
# Add/remove fields here if needed.
#
# player_rank already contains -1 when unavailable.
# player_rank_available tells the model whether -1 is missing.
#
# opponent_difficulty_score already contains -1 when
# no opponent rank is available.
# opponent_difficulty_confidence gives coverage.
# ============================================================

previous_tournament_features = [
    "tournament_finish_score",
    "player_rank",
    "player_rank_available",
    "tournament_level_score",
    "opponent_difficulty_score",
    "opponent_difficulty_confidence",
    "is_hard_surface",
    "is_clay_surface",
    "is_spring",
    "is_summer",
    "is_winter",
]


# ============================================================
# 13. CREATE prev1 / prev2 / prev3
# ============================================================

grouped = df.groupby("player_key", sort=False)

for lag in range(1, 4):
    # Keep tournament identity too
    df[f"prev{lag}_tournament_key"] = grouped["tournament_key"].shift(lag)
    df[f"prev{lag}_tourney_name"] = grouped["tourney_name"].shift(lag)
    df[f"prev{lag}_tourney_date"] = grouped["tourney_date"].shift(lag)

    # Add previous tournament features
    for feature in previous_tournament_features:
        df[f"prev{lag}_{feature}"] = grouped[feature].shift(lag)

# ============================================================
# 14. VALIDATE THE 3 PREVIOUS TOURNAMENTS
# ============================================================
# -1 marks a missing rank in an existing previous tournament
# Missing rank in an EXISTING previous tournament is -1,
# with prev*_player_rank_available = 0.
# ============================================================

for lag in range(1, 4):
    tournament_exists = df[f"prev{lag}_tournament_key"].notna()
    # Existing previous tournament should have a rank value
    # (-1 is valid and means unavailable)
    bad_rank = df[tournament_exists & df[f"prev{lag}_player_rank"].isna()]

    if not bad_rank.empty:
        raise ValueError(f"prev{lag}: existing tournament has NaN player_rank.")

    # Existing previous tournament should have availability flag
    bad_available = df[tournament_exists & df[f"prev{lag}_player_rank_available"].isna()]

    if not bad_available.empty:
        raise ValueError(f"prev{lag}: existing tournament has missing rank availability.")

    # If rank unavailable -> rank must equal -1
    bad_sentinel = df[tournament_exists & (df[f"prev{lag}_player_rank_available"] == 0) & (df[f"prev{lag}_player_rank"] != -1)]

    if not bad_sentinel.empty:
        raise ValueError(f"prev{lag}: unavailable rank is not encoded as -1.")

# ============================================================
# 15. SUMMARY FOR INTERVIEW ROWS
# ============================================================
interview_rows = df[df["has_pre_match_interview"]].copy()
eligible_interviews = interview_rows[interview_rows["num_previous_tournaments"] >= 3].copy()

print("\n" + "=" * 70)
print("FINAL HISTORY VALIDATION")
print("=" * 70)
print("Rows in tournament dataset:", len(df))
print("Rows with pre-tournament interview:", len(interview_rows))
print("Interview rows with at least 3 previous tournaments:", len(eligible_interviews))
print("Interview rows with fewer than 3 previous tournaments:", (interview_rows["num_previous_tournaments"] < 3).sum())
print("\nMissing current player rank among eligible interviews:")
print((eligible_interviews["player_rank_available"] == 0).sum())

for lag in range(1, 4):
    print(f"Missing rank in prev{lag} among eligible interviews:", (eligible_interviews[f"prev{lag}_player_rank_available"] == 0).sum())

print("\nOpponent difficulty coverage in previous tournaments:")

for lag in range(1, 4):
    print(f"\nprev{lag}:")
    print(eligible_interviews[f"prev{lag}_opponent_difficulty_confidence"].describe())

# ============================================================
# 16. SAVE UPDATED FEATURE DATAFRAME
# ============================================================

player_tournament_interview_features_df = df

In [ ]:
player_tournament_interview_features_df.to_csv(FEATURES_DATASET_FILE, index=False)

print("Saved feature dataset to:")
print(FEATURES_DATASET_FILE)

### Performance Improvement Target

The binary target compares the player's performance in the current tournament with the average performance across the three previous tournaments.

The target is defined only when all three previous tournament finish scores are available, so the historical average is never calculated from only one or two previous tournaments.

In [ ]:
# ============================================================
# Current performance compared with previous 3 tournaments
# ============================================================

previous_finish_columns = [
    "prev1_tournament_finish_score",
    "prev2_tournament_finish_score",
    "prev3_tournament_finish_score",
]

has_three_previous_finish_scores = player_tournament_interview_features_df[previous_finish_columns].notna().all(axis=1)
previous_three_finish_average = player_tournament_interview_features_df[previous_finish_columns].mean(axis=1)

player_tournament_interview_features_df["current_finish_at_least_recent_average"] = np.where(
    has_three_previous_finish_scores,
    (player_tournament_interview_features_df["tournament_finish_score"] >= previous_three_finish_average).astype(int), np.nan)

### Historical Feature Validation

The constructed history features are validated before selecting the final interview rows. In particular, we verify the availability of three previous tournaments and inspect missing historical information for interview observations.

In [ ]:
# ============================================================
# Validation of historical features
# ============================================================

interview_rows_mask = player_tournament_interview_features_df["has_pre_match_interview"]

print("Interview rows:", int(interview_rows_mask.sum()))

print("Interview rows with fewer than 3 previous tournaments:",
    int((interview_rows_mask & player_tournament_interview_features_df["prev3_tournament_key"].isna()).sum()))

print("Interview rows with no prev1 opponent-rank information:",
    int((interview_rows_mask & (player_tournament_interview_features_df["prev1_opponent_difficulty_confidence"] == 0)).sum()))

print("Interview rows with no prev2 opponent-rank information:",
    int((interview_rows_mask & (player_tournament_interview_features_df["prev2_opponent_difficulty_confidence"] == 0)).sum()))

print("Interview rows with no prev3 opponent-rank information:",
    int((interview_rows_mask & (player_tournament_interview_features_df["prev3_opponent_difficulty_confidence"] == 0)).sum()))

print("Interview rows where binary performance target could not be calculated:",
    int((interview_rows_mask & player_tournament_interview_features_df["current_finish_at_least_recent_average"].isna()).sum()))

In [ ]:
# ============================================================
# Build final interview dataset
# Keep only interviews with 3 previous tournaments
# ============================================================

final_interview_df = (
    player_tournament_interview_features_df[
        player_tournament_interview_features_df["has_pre_match_interview"]
        & (player_tournament_interview_features_df["num_previous_tournaments"] >= 3)
    ].copy().reset_index(drop=True)
)

print("Final interview rows:", len(final_interview_df))
print("Missing target values:", final_interview_df["current_finish_at_least_recent_average"].isna().sum())

In [ ]:
# ============================================================
# Missing values report - final interview dataset
# ============================================================

missing_values_df = pd.DataFrame({"missing_count": final_interview_df.isna().sum()})

missing_values_df["missing_percent"] = (
    100
    * missing_values_df["missing_count"]
    / len(final_interview_df)
)

missing_values_df = (
    missing_values_df
    .sort_values(
        "missing_count",
        ascending=False
    )
)

print("Columns with missing values:", (missing_values_df["missing_count"] > 0).sum())
display(missing_values_df[missing_values_df["missing_count"] > 0])

In [ ]:
# ============================================================
# Remove helper columns
# ============================================================

columns_to_drop = [
    "prev1_tournament_key",
    "prev2_tournament_key",
    "prev3_tournament_key",

    "prev1_tourney_name",
    "prev2_tourney_name",
    "prev3_tourney_name",

    "prev1_tourney_date",
    "prev2_tourney_date",
    "prev3_tourney_date",
]

player_tournament_interview_features_df = player_tournament_interview_features_df.drop(columns=columns_to_drop, errors="ignore")
final_interview_df = final_interview_df.drop(columns=columns_to_drop, errors="ignore")

# ============================================================
# Save full tournament dataset
# ============================================================
player_tournament_interview_features_df.to_csv(PREV3_FULL_DATASET_FILE, index=False)
print(f"Saved full dataset to: {PREV3_FULL_DATASET_FILE}")

In [ ]:
# ============================================================
# Prepare interviews-only dataset
# ============================================================

interviews_only_df = final_interview_df.copy()
print("Rows with interview:", len(interviews_only_df))

# ============================================================
# Remove columns not needed for the interview-level dataset
# ============================================================

columns_to_drop = [
    "player_key",
    "player_id",
    "tournament_key",
    "tourney_id",
    "draw_size",
    "tournament_rounds",
    "number_of_rounds",
    "player_hand",
    "first_round_played",
    "last_round_played",
    "matches_played",
    "matches_won",
    "matches_lost",
    "won_tournament",
    "tournament_finish_position",
    "finish_stage_index",
    "has_any_interview",
    "has_pre_match_interview",
    "num_opponents",
    "num_ranked_opponents",
    "num_previous_tournaments",

    # Previous-tournament helper identifiers
    "prev1_tournament_key",
    "prev2_tournament_key",
    "prev3_tournament_key",
]

interviews_only_df = interviews_only_df.drop(columns=columns_to_drop, errors="ignore")

print("Remaining columns:", len(interviews_only_df.columns))
print(interviews_only_df.columns.tolist())

# ============================================================
# Save interviews-only dataset
# ============================================================
interviews_only_df.to_csv(INTERVIEWS_ONLY_FILE, index=False)
print(f"Saved to: {INTERVIEWS_ONLY_FILE}")
print("Final rows:", len(interviews_only_df))

## Interview Q&A Validation and Repair

The stored question-answer JSON was validated for malformed JSON, missing question-answer pairs, unexpected keys, discontinuous numbering, and empty question or answer fields. The only issue identified was eight empty question fields across eight interviews.

These missing questions were recovered from the original interview URLs. In addition, one interview that had been incorrectly classified as pre-tournament during collection was identified and removed before calculating the final interview features.

In [ ]:
interviews_only_df = pd.read_csv(
    INTERVIEWS_ONLY_FILE,
    parse_dates=[
        "tourney_date",
        "first_pre_match_interview_date",
    ]
)

JSON_COL = "first_pre_match_interview_qa_json"

interviews_df = interviews_only_df[interviews_only_df[JSON_COL].notna()].copy()
issues = []

for row_idx, row in interviews_df.iterrows():
    raw_json = row[JSON_COL]
    try:
        qa = json.loads(raw_json)
    except Exception as e:
        issues.append({
            "row_index": row_idx,
            "player": row["player_name"],
            "tournament": row["tourney_name"],
            "year": row["tourney_year"],
            "issue": "invalid_json",
            "details": str(e),
        })
        continue
    keys = list(qa.keys())

    # ---------------------------------------------------------
    # 1. Check for unexpected keys
    # ---------------------------------------------------------
    unexpected_keys = [key for key in keys if not re.fullmatch(r"(question|answer)_\d+", key)]

    if unexpected_keys:
        issues.append({
            "row_index": row_idx,
            "player": row["player_name"],
            "tournament": row["tourney_name"],
            "year": row["tourney_year"],
            "issue": "unexpected_keys",
            "details": unexpected_keys,
        })

    # ---------------------------------------------------------
    # 2. Extract question and answer indices
    # ---------------------------------------------------------
    question_indices = {
        int(match.group(1))
        for key in keys
        if (match := re.fullmatch(r"question_(\d+)", key))
    }

    answer_indices = {
        int(match.group(1))
        for key in keys
        if (match := re.fullmatch(r"answer_(\d+)", key))
    }

    # Questions without matching answers
    for i in sorted(question_indices - answer_indices):
        issues.append({
            "row_index": row_idx,
            "player": row["player_name"],
            "tournament": row["tourney_name"],
            "year": row["tourney_year"],
            "issue": "question_without_answer",
            "details": f"question_{i}",
        })

    # Answers without matching questions
    for i in sorted(answer_indices - question_indices):
        issues.append({
            "row_index": row_idx,
            "player": row["player_name"],
            "tournament": row["tourney_name"],
            "year": row["tourney_year"],
            "issue": "answer_without_question",
            "details": f"answer_{i}",
        })

    # ---------------------------------------------------------
    # 3. Check that numbering is continuous: 1, 2, 3, ...
    # ---------------------------------------------------------
    all_indices = question_indices | answer_indices

    if all_indices:
        expected_indices = set(range(1, max(all_indices) + 1))
        missing_indices = sorted(expected_indices - all_indices)

        if missing_indices:
            issues.append({
                "row_index": row_idx,
                "player": row["player_name"],
                "tournament": row["tourney_name"],
                "year": row["tourney_year"],
                "issue": "missing_index",
                "details": missing_indices,
            })

    # ---------------------------------------------------------
    # 4. Check actual key order
    # Expected:
    # question_1, answer_1, question_2, answer_2, ...
    # ---------------------------------------------------------
    expected_key_order = []

    if all_indices:
        for i in range(1, max(all_indices) + 1):
            if i in question_indices:
                expected_key_order.append(f"question_{i}")
            if i in answer_indices:
                expected_key_order.append(f"answer_{i}")

        relevant_keys = [key for key in keys if re.fullmatch(r"(question|answer)_\d+", key)]

        if relevant_keys != expected_key_order:
            issues.append({
                "row_index": row_idx,
                "player": row["player_name"],
                "tournament": row["tourney_name"],
                "year": row["tourney_year"],
                "issue": "unexpected_key_order",
                "details": relevant_keys,
            })


issues_df = pd.DataFrame(issues)

print("Number of interviews checked:", len(interviews_df))
print("Number of interviews with any issue:", issues_df["row_index"].nunique() if not issues_df.empty else 0)

if issues_df.empty:
    print("\nAll interviews have a valid Q-A structure.")
else:
    print("\nIssue counts:")
    print(issues_df["issue"].value_counts())

    display(issues_df)

In [ ]:
empty_qa = []

for row_idx, row in interviews_df.iterrows():
    qa = json.loads(row[JSON_COL])

    for key, value in qa.items():
        # Missing, non-string, or empty after removing whitespace
        if not isinstance(value, str) or not value.strip():
            empty_qa.append({
                "row_index": row_idx,
                "player": row["player_name"],
                "tournament": row["tourney_name"],
                "year": row["tourney_year"],
                "key": key,
                "value": value,
            })

empty_qa_df = pd.DataFrame(empty_qa)

print("Number of interviews checked:", len(interviews_df))
print("Number of empty/invalid Q-A fields:", len(empty_qa_df))

if empty_qa_df.empty:
    print("All questions and answers contain text.")
else:
    display(empty_qa_df)

In [ ]:
for _, issue in empty_qa_df.iterrows():

    row_idx = issue["row_index"]
    key = issue["key"]

    row = interviews_df.loc[row_idx]
    qa = json.loads(row[JSON_COL])

    i = int(key.split("_")[1])

    print("\n" + "=" * 100)
    print(
        f'{row["player_name"]} | '
        f'{row["tourney_name"]} | '
        f'{row["tourney_year"]}'
    )
    print("URL:", row["first_pre_match_interview_url"])
    print("=" * 100)

    # Show previous, problematic, and next Q-A pair
    for j in range(max(1, i - 1), i + 2):

        question_key = f"question_{j}"
        answer_key = f"answer_{j}"

        if question_key not in qa and answer_key not in qa:
            continue

        marker = "  <-- PROBLEM" if j == i else ""

        print(f"\nPAIR {j}{marker}")

        print("QUESTION:")
        print(repr(qa.get(question_key)))

        print("\nANSWER:")
        print(repr(qa.get(answer_key)))

In [ ]:
# ============================================================
# Validate interview Q&A JSON structure
# ============================================================

QA_COLUMN = "first_pre_match_interview_qa_json"
validation_issues = []
valid_key_pattern = re.compile(r"^(question|answer)_(\d+)$")


for idx, row in interviews_only_df.iterrows():
    qa_json = row[QA_COLUMN]

    row_context = {
        "row_index": idx,
        "player_name": row["player_name"],
        "tourney_name": row["tourney_name"],
        "tourney_year": row["tourney_year"],
    }

    # --------------------------------------------------------
    # Missing JSON
    # --------------------------------------------------------

    if pd.isna(qa_json) or not str(qa_json).strip():
        validation_issues.append({
            **row_context,
            "issue_type": "missing_json",
            "details": "Q&A JSON is missing or empty",
        })
        continue

    # --------------------------------------------------------
    # Malformed JSON
    # --------------------------------------------------------

    try:
        qa = json.loads(qa_json)
    except json.JSONDecodeError as exp:
        validation_issues.append({
            **row_context,
            "issue_type": "malformed_json",
            "details": str(exp),
        })
        continue

    # --------------------------------------------------------
    # JSON must contain a dictionary
    # --------------------------------------------------------

    if not isinstance(qa, dict):
        validation_issues.append({
            **row_context,
            "issue_type": "unexpected_json_type",
            "details": f"Expected dict, got {type(qa).__name__}",
        })
        continue

    question_numbers = set()
    answer_numbers = set()

    # --------------------------------------------------------
    # Unexpected keys + empty fields
    # --------------------------------------------------------

    for key, value in qa.items():
        match = valid_key_pattern.fullmatch(str(key))
        if not match:
            validation_issues.append({
                **row_context,
                "issue_type": "unexpected_key",
                "details": str(key),
            })

            continue

        field_type = match.group(1)
        number = int(match.group(2))

        if field_type == "question":
            question_numbers.add(number)
        else:
            answer_numbers.add(number)

        if value is None or not str(value).strip():

            validation_issues.append({
                **row_context,
                "issue_type": (
                    "empty_question"
                    if field_type == "question"
                    else "empty_answer"
                ),
                "details": key,
            })

    # --------------------------------------------------------
    # Missing question-answer pairs
    # --------------------------------------------------------

    for number in sorted(question_numbers - answer_numbers):
        validation_issues.append({
            **row_context,
            "issue_type": "missing_answer",
            "details": f"answer_{number}",
        })

    for number in sorted(answer_numbers - question_numbers):
        validation_issues.append({
            **row_context,
            "issue_type": "missing_question",
            "details": f"question_{number}",
        })

    # --------------------------------------------------------
    # Discontinuous numbering
    # --------------------------------------------------------

    all_numbers = (question_numbers | answer_numbers)

    if all_numbers:
        expected_numbers = set(range(1, max(all_numbers) + 1))
        missing_numbers = sorted(expected_numbers - all_numbers)

        if missing_numbers:
            validation_issues.append({
                **row_context,
                "issue_type": "discontinuous_numbering",
                "details": (
                    "Missing pair numbers: "
                    + ", ".join(
                        map(str, missing_numbers)
                    )
                ),
            })


# ============================================================
# Validation summary
# ============================================================

qa_validation_df = pd.DataFrame(validation_issues)

print("=" * 70)
print("INTERVIEW Q&A JSON VALIDATION")
print("=" * 70)
print("Interviews checked:", len(interviews_only_df))

if qa_validation_df.empty:
    print("No Q&A JSON issues found.")
else:
    print("Interviews with at least one issue:", qa_validation_df["row_index"].nunique())
    print("\nIssues by type:")
    display(qa_validation_df["issue_type"].value_counts().rename_axis("issue_type").reset_index(name="count"))

    print("\nIssue details:")

    display(qa_validation_df.sort_values(
            [
                "issue_type",
                "player_name",
                "tourney_year",
            ]
        )
    )

In [ ]:
# Repair the empty questions identified above using the original
# interview URLs and remove the interview incorrectly classified
# as pre-tournament.

# ============================================================
# Settings
# ============================================================

REQUEST_DELAY_SECONDS = 0.2
session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/149.0.0.0 Safari/537.36"
        )
    }
)


# ============================================================
# Remove an interview incorrectly classified as pre-tournament
# ============================================================

NON_PRE_TOURNAMENT_URLS_TO_REMOVE = {
    "http://www.asapsports.com/show_interview.php?id=190664",
}

non_pre_tournament_mask = interviews_only_df["first_pre_match_interview_url"].isin(NON_PRE_TOURNAMENT_URLS_TO_REMOVE)

print("Known non-pre-tournament interviews to remove:", int(non_pre_tournament_mask.sum()))

if non_pre_tournament_mask.any():
    display(
        interviews_only_df.loc[
            non_pre_tournament_mask,
            [
                "player_name",
                "tourney_name",
                "tourney_year",
                "first_pre_match_interview_url",
            ]
        ]
    )

interviews_only_df = interviews_only_df.loc[~non_pre_tournament_mask].reset_index(drop=True)

print("Interview rows after removing non-pre-tournament interviews:", len(interviews_only_df))


# ============================================================
# Basic text helpers
# ============================================================

def clean_text(value):
    return re.sub(r"\s+", " ", str(value)).strip()


def get_all_lines(soup):
    text = soup.get_text("\n")
    return [line.strip() for line in text.splitlines() if line.strip()]

# ============================================================
# Download interview page
# ============================================================

def fetch_interview_soup(url):
    time.sleep(REQUEST_DELAY_SECONDS)
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

# ============================================================
# Detect question / answer starts
# ============================================================

def is_question_start(line):
    line = line.strip()
    return (line.startswith("Q.") or line.startswith("THE MODERATOR:"))

def should_ignore_moderator_line(line):
    normalized = clean_text(line).lower()
    return normalized.startswith("the moderator: questions")

def is_player_answer_start(line, player_name):
    line_upper = line.strip().upper()
    player_upper = player_name.strip().upper()
    return line_upper.startswith(player_upper + ":")

def remove_question_prefix(line):
    line = line.strip()
    if line.startswith("Q."):
        return re.sub(r"^Q\.\s*", "", line).strip()

    if line.startswith("THE MODERATOR:"):
        return re.sub(
            r"^THE MODERATOR:\s*",
            "",
            line
        ).strip()
    return line


def remove_answer_prefix(line, player_name):
    return re.sub(
        rf"^{re.escape(player_name)}:\s*",
        "",
        line.strip(),
        flags=re.IGNORECASE,
    ).strip()


# ============================================================
# Extract Q&A from original page
#
# IMPORTANT:
# Keep lines that may contain score-like text
# A real question may contain strings such as:
# 6-4, 7-5, 6-1, etc.
# ============================================================

def extract_qa_from_original_page(soup, player_name):
    lines = get_all_lines(soup)

    # --------------------------------------------------------
    # Find interview content boundaries
    # --------------------------------------------------------

    start_index = 0
    end_index = len(lines)

    # Start after "Press Conference" if it exists
    for i, line in enumerate(lines):
        if (line.lower() == "press conference"):
            start_index = i + 1
            break

    # Stop before ASAP footer
    for i in range(start_index, len(lines)):
        if ("FastScripts Transcript by ASAP Sports" in lines[i]):
            end_index = i
            break

    content_lines = lines[start_index:end_index]

    # --------------------------------------------------------
    # Parse question-answer pairs
    # --------------------------------------------------------
    qa = {}
    question_counter = 0
    current_question_lines = []
    current_answer_lines = []
    state = "searching_question"

    for line in content_lines:
        # Ignore generic moderator heading
        if should_ignore_moderator_line(line):
            continue

        # --------------------------------------------
        # Searching for beginning of a question
        # --------------------------------------------
        if state == "searching_question":
            if is_question_start(line):
                question_text = remove_question_prefix(line)
                current_question_lines = [question_text] if question_text else []
                current_answer_lines = []
                state = "reading_question"
            continue

        # --------------------------------------------
        # Reading question
        # --------------------------------------------
        if state == "reading_question":
            if is_player_answer_start(line, player_name):
                answer_text = remove_answer_prefix(line, player_name)
                current_answer_lines = [answer_text] if answer_text else []
                state = "reading_answer"
            else:
                current_question_lines.append(line)
            continue

        # --------------------------------------------
        # Reading answer
        # --------------------------------------------

        if state == "reading_answer":
            if is_question_start(line):
                # Save previous pair
                question_counter += 1
                qa[f"question_{question_counter}"] = clean_text(" ".join(current_question_lines))
                qa[f"answer_{question_counter}"] = clean_text(" ".join(current_answer_lines))

                # Start next question
                question_text = remove_question_prefix(line)

                current_question_lines = [question_text] if question_text else []
                current_answer_lines = []
                state = "reading_question"
            else:
                current_answer_lines.append(line)

    # --------------------------------------------------------
    # Save final pair
    # --------------------------------------------------------
    if (current_question_lines and current_answer_lines):
        question_counter += 1
        qa[f"question_{question_counter}"] = clean_text(" ".join(current_question_lines))
        qa[f"answer_{question_counter}"] = clean_text(" ".join(current_answer_lines))
    return qa


# ============================================================
# Find interviews containing empty questions
# ============================================================

def get_empty_question_keys(qa_json):

    if pd.isna(qa_json):
        return []
    try:
        qa = json.loads(qa_json)
    except json.JSONDecodeError:
        return []

    return [
        key
        for key, value in qa.items()
        if (
            key.startswith(
                "question_"
            )
            and not str(
                value
            ).strip()
        )
    ]


interviews_only_df["_empty_question_keys"] = interviews_only_df["first_pre_match_interview_qa_json"].apply(get_empty_question_keys)

problem_mask = interviews_only_df["_empty_question_keys"].apply(len) > 0

print("\nInterviews with empty questions:", int(problem_mask.sum()))

print("Total empty questions:",
    int(
        interviews_only_df.loc[
            problem_mask,
            "_empty_question_keys"
        ]
        .apply(len)
        .sum()
    )
)


# ============================================================
# Repair ONLY the empty question fields
# ============================================================

repair_log = []
for idx in interviews_only_df.index[problem_mask]:
    row = interviews_only_df.loc[idx]

    player_name = row["player_name"]
    url = row["first_pre_match_interview_url"]
    empty_keys = row["_empty_question_keys"]

    print(
        f"\nRepairing: "
        f"{player_name} | "
        f"{row['tourney_name']} | "
        f"{row['tourney_year']}"
    )

    print("Missing:", empty_keys)
    print("URL:", url)

    try:
        # --------------------------------------------
        # Existing JSON
        # --------------------------------------------
        current_qa = json.loads(row["first_pre_match_interview_qa_json"])

        # --------------------------------------------
        # Download original interview
        # --------------------------------------------
        soup = fetch_interview_soup(url)
        repaired_qa = extract_qa_from_original_page(soup, player_name)

        # --------------------------------------------
        # Replace ONLY fields that were empty
        # --------------------------------------------

        for question_key in empty_keys:
            original_value = current_qa.get(question_key, "")
            repaired_value = repaired_qa.get(question_key, "")

            if str(repaired_value).strip():
                current_qa[question_key] = repaired_value
                print(f"  Fixed {question_key}:")
                print(f"  {repaired_value}")

                repair_log.append(
                    {
                        "row_index":
                            idx,
                        "player_name":
                            player_name,
                        "tourney_name":
                            row[
                                "tourney_name"
                            ],
                        "tourney_year":
                            row[
                                "tourney_year"
                            ],
                        "question_key":
                            question_key,
                        "old_value":
                            original_value,
                        "new_value":
                            repaired_value,
                        "status":
                            "repaired",
                    }
                )

            else:
                print(f"  Could not recover {question_key}")

                repair_log.append(
                    {
                        "row_index":
                            idx,
                        "player_name":
                            player_name,
                        "tourney_name":
                            row[
                                "tourney_name"
                            ],
                        "tourney_year":
                            row[
                                "tourney_year"
                            ],
                        "question_key":
                            question_key,
                        "old_value":
                            original_value,
                        "new_value":
                            repaired_value,
                        "status":
                            "not_repaired",
                    }
                )

        # --------------------------------------------
        # Save JSON back to ONLY this row
        # --------------------------------------------

        interviews_only_df.at[idx, "first_pre_match_interview_qa_json"] = json.dumps(current_qa, ensure_ascii=False)

    except Exception as exp:
        print(f"  ERROR: {exp}")

        for question_key in empty_keys:
            repair_log.append(
                {
                    "row_index":
                        idx,
                    "player_name":
                        player_name,
                    "tourney_name":
                        row[
                            "tourney_name"
                        ],
                    "tourney_year":
                        row[
                            "tourney_year"
                        ],
                    "question_key":
                        question_key,
                    "old_value":
                        "",
                    "new_value":
                        "",
                    "status":
                        f"error: {exp}",
                }
            )


# ============================================================
# Remove temporary helper column
# ============================================================

interviews_only_df = interviews_only_df.drop(columns=["_empty_question_keys"])

# ============================================================
# Validation
# ============================================================

remaining_empty_questions = []

for idx, row in (interviews_only_df.iterrows()):
    qa_json = row["first_pre_match_interview_qa_json"]

    if pd.isna(qa_json):
        continue
    try:
        qa = json.loads(qa_json)
    except json.JSONDecodeError:
        continue

    for key, value in qa.items():
        if (key.startswith("question_") and not str( value).strip()):
            remaining_empty_questions.append(
                {
                    "row_index":
                        idx,
                    "player_name":
                        row[
                            "player_name"
                        ],
                    "tourney_name":
                        row[
                            "tourney_name"
                        ],
                    "tourney_year":
                        row[
                            "tourney_year"
                        ],
                    "question_key":
                        key,
                }
            )


print("\nRemaining empty questions:", len(remaining_empty_questions))

print("Final number of interviews:", len(interviews_only_df))


# ============================================================
# Show exactly what was changed
# ============================================================

repair_log_df = (
    pd.DataFrame(
        repair_log
    )
)

display(repair_log_df)

In [ ]:
interviews_only_df.to_csv(INTERVIEWS_FIXED_FILE, index=False)

print(f"Saved repaired dataset to: {INTERVIEWS_FIXED_FILE}")

## Interview Structure Features

Three structural features are extracted from each validated pre-tournament interview: the number of valid question-answer pairs, the total number of answer words, and the average answer length in words.

In [ ]:
# ============================================================
# Interview structure features
# ============================================================

interviews_only_df = pd.read_csv(
    INTERVIEWS_FIXED_FILE,
    parse_dates=[
        "tourney_date",
        "first_pre_match_interview_date",
    ]
)

def extract_interview_features(qa_json):
    if pd.isna(qa_json):
        return pd.Series(
            {
                "num_questions": np.nan,
                "avg_answer_length_words": np.nan,
                "total_answer_words": np.nan,
            }
        )

    if isinstance(qa_json, str):
        try:
            qa = json.loads(qa_json)
        except json.JSONDecodeError:
            return pd.Series(
                {
                    "num_questions": np.nan,
                    "avg_answer_length_words": np.nan,
                    "total_answer_words": np.nan,
                }
            )
    else:
        qa = qa_json

    num_question_answer_pairs = 0
    answer_lengths = []

    i = 1

    while True:

        question = qa.get(f"question_{i}")
        answer = qa.get(f"answer_{i}")

        if question is None and answer is None:
            break

        if (
            question is not None
            and str(question).strip()
            and answer is not None
            and str(answer).strip()
        ):
            num_question_answer_pairs += 1

            num_words = len(
                re.findall(
                    r"\S+",
                    str(answer)
                )
            )

            answer_lengths.append(num_words)

        i += 1

    if answer_lengths:
        avg_answer_length = np.mean(answer_lengths)
        total_answer_words = np.sum(answer_lengths)
    else:
        avg_answer_length = np.nan
        total_answer_words = np.nan

    return pd.Series(
        {
            "num_questions": num_question_answer_pairs,
            "avg_answer_length_words": avg_answer_length,
            "total_answer_words": total_answer_words,
        }
    )


interviews_only_df[
    [
        "num_questions",
        "avg_answer_length_words",
        "total_answer_words",
    ]
] = (
    interviews_only_df[
        "first_pre_match_interview_qa_json"
    ]
    .apply(extract_interview_features)
)

In [ ]:
interviews_only_df.to_csv(
    FINAL_INTERVIEWS_FILE,
    index=False,
)

print(f"Saved dataset to: {FINAL_INTERVIEWS_FILE}")

## Final Dataset Validation

The final interview-level dataset is checked for missing values and numeric ranges.

In [ ]:
# Missing values summary
missing_summary = pd.DataFrame({
    "missing_count": interviews_only_df.isna().sum(),
    "missing_percent": (
        interviews_only_df.isna().mean() * 100
    ).round(2),
    "unique_values": interviews_only_df.nunique(dropna=False),
})

missing_summary = (
    missing_summary
    .sort_values(
        ["missing_count", "unique_values"],
        ascending=[False, True]
    )
)

display(missing_summary)

In [ ]:
display(
    interviews_only_df
    .select_dtypes(include="number")
    .describe()
    .T
)

In [ ]:
shutil.copy2(FINAL_INTERVIEWS_FILE, FINAL_DATASET)
print(f"Final dataset copied to: {FINAL_DATASET}")